# AFRICA GIANTS — Accuracy Gate Eval

Loads base model + LoRA adapter, runs inference on 200 eval questions, scores by subdomain.

In [ ]:
import os, json, re, sys, subprocess, urllib.request
from datetime import datetime, timezone

# Auth — Kaggle secrets with env fallback
try:
    import kaggle_secrets
    us = kaggle_secrets.UserSecretsClient()
    hf_token = us.get_secret("AFRICA_GIANTS")
    print(f"[auth] HF token loaded ({hf_token[:8]}...)")
except Exception as _e:
    hf_token = os.environ.get("HF_TOKEN", "")
    print(f"[auth] fallback env HF_TOKEN: {hf_token[:8] if hf_token else 'MISSING'}")

# Load chike_config.json from GitHub (single source of truth)
_CONFIG_URL = (
    "https://raw.githubusercontent.com/"
    "prosperpiusmbaruku007-ship-it/AFRICA-GIANTS/main/kaggle/chike_config.json"
)
try:
    with urllib.request.urlopen(_CONFIG_URL, timeout=15) as _resp:
        _cfg = json.loads(_resp.read().decode())
    print(f"[config] chike_config.json loaded from GitHub (version={_cfg.get('version','?')})")
except Exception as _cfg_err:
    print(f"[config] WARNING: GitHub fetch failed: {_cfg_err} — check network")
    _cfg = {}

SYSTEM_PROMPT      = _cfg.get("system_prompt", "")
MAX_NEW_TOKENS     = _cfg.get("generation", {}).get("max_new_tokens", 256)
ACCURACY_THRESHOLD = _cfg.get("thresholds", {}).get("accuracy", 0.85)
REFUSAL_THRESHOLD  = _cfg.get("thresholds", {}).get("refusal", 0.70)

# Version-specific — update each eval run
BASE_MODEL   = "McGill-NLP/AfriqueLlama-8B"
ADAPTER_REPO = "prospAprospA007/africa-giants-adapter-v12"

print(f"[config] SYSTEM_PROMPT: {len(SYSTEM_PROMPT)} chars")
print(f"[config] MAX_NEW_TOKENS={MAX_NEW_TOKENS}  ACC={ACCURACY_THRESHOLD}  REF={REFUSAL_THRESHOLD}")
print(f"[config] ADAPTER_REPO={ADAPTER_REPO}")
print("[config] done")


In [ ]:
ret = subprocess.run(
    "pip install -q 'transformers>=4.43.0' peft accelerate bitsandbytes 2>&1 | tail -5",
    shell=True, capture_output=True, text=True
)
print(ret.stdout or ret.stderr)


In [ ]:
import torch
print(f"[gpu] CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"[gpu] {p.name}  sm_{p.major}{p.minor}  {p.total_memory/1e9:.1f}GB")


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

print("[model] Loading base model + adapter ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=hf_token, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto",
    token=hf_token, trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO, token=hf_token)
model.eval()
print("[model] loaded OK")


In [ ]:
eval_questions = json.loads('[{"id": "eval_001", "subdomain": "vat_registration", "question_sw": "Kiwango cha VAT Tanzania kwa bidhaa na huduma za kawaida ni asilimia ngapi?", "question_en": "What is the standard VAT rate in Tanzania?", "correct_answer_sw": "Asilimia 18. Kiwango hiki kinahusu bidhaa na huduma zote zinazolipwa VAT nchini Tanzania Bara. Thibitisha na TRA baada ya kila Finance Act.", "correct_answer_en": "18%. This rate applies to all taxable goods and services on Tanzania Mainland. Confirm with TRA after each Finance Act.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_002", "subdomain": "vat_registration", "question_sw": "Kizingiti cha mauzo cha miezi 12 cha kusajilisha VAT kwa lazima ni TZS ngapi?", "question_en": "What is the mandatory VAT registration threshold for a rolling 12-month period?", "correct_answer_sw": "TZS milioni 200 katika kipindi chochote cha miezi 12 mfululizo. Ukizidi kiasi hiki, usajilishaji wa VAT ni wa lazima mara moja. Thibitisha na TRA.", "correct_answer_en": "TZS 200 million in any consecutive 12-month period. Once exceeded, VAT registration is immediately mandatory. Confirm with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_003", "subdomain": "vat_registration", "question_sw": "Kizingiti cha mauzo cha miezi 6 cha kusajilisha VAT kwa lazima ni TZS ngapi?", "question_en": "What is the mandatory VAT registration threshold for a rolling 6-month period?", "correct_answer_sw": "TZS milioni 100 katika kipindi chochote cha miezi 6 mfululizo. Hii ni kizingiti cha ziada \\u2014 ukizidi hata moja ya vizingiti viwili, lazima ujisajilishe. Thibitisha na TRA.", "correct_answer_en": "TZS 100 million in any consecutive 6-month period. This is an alternative threshold \\u2014 exceeding either triggers mandatory registration. Confirm with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_004", "subdomain": "vat_registration", "question_sw": "Kiwango cha VAT kwa malipo ya kidijitali ya B2C (biashara-kwa-mteja) tangu 1 Septemba 2025 ni asilimia ngapi?", "question_en": "What is the B2C digital payment VAT rate effective 1 September 2025?", "correct_answer_sw": "Asilimia 16 kuanzia 1 Septemba 2025. Hata hivyo, kanuni za utekelezaji bado zinasubiri tangazo rasmi la Kamishna Mkuu (CG). Usitekeleze bila uthibitisho wa TRA.", "correct_answer_en": "16% from 1 September 2025. However, implementation rules are still pending the Commissioner General\'s formal notice. Do not apply without TRA confirmation.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_005", "subdomain": "vat_registration", "question_sw": "Deadline ya kuwasilisha VAT return kila mwezi ni tarehe ngapi?", "question_en": "What is the monthly VAT return filing deadline?", "correct_answer_sw": "Siku ya 20 ya mwezi unaofuata. Kwa mfano, VAT return ya Aprili iwasilishwe ifikapo 20 Mei. Ucheleweshaji unasababisha faini na riba. Thibitisha na TRA.", "correct_answer_en": "The 20th of the following month. For example, April\'s VAT return is due by 20 May. Late filing incurs penalties and interest. Confirm with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_006", "subdomain": "vat_registration", "question_sw": "Biashara inaingiza TZS milioni 25 kwa mwezi \\u2014 baada ya miezi mingapi inafika kizingiti cha VAT cha TZS milioni 200?", "question_en": "A business earns TZS 25 million per month \\u2014 after how many months does it hit the TZS 200 million VAT threshold?", "correct_answer_sw": "Baada ya miezi 8 (25M \\u00d7 8 = 200M). Lazima ujisajilishe mara mauzo yakifika kizingiti \\u2014 usiendelee bila usajilishaji. Thibitisha tarehe ya kuzidi na TRA.", "correct_answer_en": "After 8 months (25M \\u00d7 8 = 200M). You must register as soon as sales reach the threshold \\u2014 do not continue unregistered. Confirm the crossing date with TRA.", "answer_type": "number", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_007", "subdomain": "vat_registration", "question_sw": "Biashara inapata TZS milioni 20 kwa mwezi \\u2014 baada ya miezi mingapi inazidi kizingiti cha VAT cha miezi 6 (TZS milioni 100)?", "question_en": "A business earns TZS 20 million per month \\u2014 after how many months does it cross the 6-month VAT threshold of TZS 100 million?", "correct_answer_sw": "Baada ya miezi 5 (20M \\u00d7 5 = 100M). Usajilishaji wa VAT unakuwa wa lazima mara mauzo yakizidi TZS milioni 100 katika kipindi chochote cha miezi 6. Thibitisha na TRA.", "correct_answer_en": "After 5 months (20M \\u00d7 5 = 100M). VAT registration becomes mandatory once sales exceed TZS 100 million in any 6-month period. Confirm with TRA.", "answer_type": "number", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_008", "subdomain": "vat_registration", "question_sw": "VAT ya matokeo ni TZS 950,000 na VAT ya pembejeo ni TZS 320,000 \\u2014 ninapeleka kiasi gani TRA?", "question_en": "Output VAT is TZS 950,000 and input VAT is TZS 320,000 \\u2014 how much is remitted to TRA?", "correct_answer_sw": "TZS 630,000 (950,000 \\u2212 320,000). Unalipa TRA tofauti kati ya VAT ya matokeo na VAT ya pembejeo peke yake. Thibitisha hesabu na mshauri wa kodi.", "correct_answer_en": "TZS 630,000 (950,000 \\u2212 320,000). You pay TRA only the difference between output VAT and input VAT. Confirm the calculation with a tax adviser.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_009", "subdomain": "vat_registration", "question_sw": "Tofauti ya asilimia kati ya kiwango cha kawaida cha VAT (18%) na kiwango cha B2C cha kidijitali (16%) ni ngapi?", "question_en": "What is the percentage-point difference between the standard VAT rate (18%) and the B2C digital rate (16%)?", "correct_answer_sw": "Pointi 2 za asilimia (18% \\u2212 16% = 2%). Kiwango cha B2C cha asilimia 16 kilianzishwa kuanzia 1 Septemba 2025, bado kinasubiri utekelezaji kamili.", "correct_answer_en": "2 percentage points (18% \\u2212 16% = 2%). The 16% B2C rate was introduced from 1 September 2025, still pending full implementation.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_010", "subdomain": "vat_registration", "question_sw": "Biashara yangu ina mauzo ya TZS milioni 180 mwaka huu \\u2014 ninahitaji mauzo ya ziada ya TZS ngapi kabla ya usajilishaji wa VAT wa lazima?", "question_en": "My business has TZS 180 million in annual sales \\u2014 how much more revenue until mandatory VAT registration?", "correct_answer_sw": "TZS milioni 20 zaidi (200M \\u2212 180M). Lakini fuatilia pia kizingiti cha miezi 6 cha TZS milioni 100 \\u2014 inawezekana ufike kizingiti hicho mapema zaidi. Thibitisha na TRA.", "correct_answer_en": "TZS 20 million more (200M \\u2212 180M). But also monitor the 6-month threshold of TZS 100 million \\u2014 you may reach that sooner. Confirm with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_011", "subdomain": "vat_registration", "question_sw": "Finance Act ya mwaka gani ilianzisha utaratibu mpya wa VAT withholding nchini Tanzania?", "question_en": "Which Finance Act year introduced the new VAT withholding regime in Tanzania?", "correct_answer_sw": "Finance Act ya 2025, ikiwa na ufanisi kuanzia 1 Julai 2025. Kiwango ni asilimia 3 kwa bidhaa na asilimia 6 kwa huduma kwa wanunuzi wanaohitimu.", "correct_answer_en": "Finance Act 2025, effective from 1 July 2025. The rate is 3% on goods and 6% on services for qualifying buyers.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_012", "subdomain": "vat_registration", "question_sw": "Kiwango cha VAT ya B2C ya kidijitali cha asilimia 16 kilianza kutumika tarehe ngapi?", "question_en": "From what date is the 16% B2C digital VAT rate applicable?", "correct_answer_sw": "Tarehe 1 Septemba 2025. Kanuni kamili za utekelezaji bado zinasubiri tangazo la Kamishna Mkuu. Thibitisha hali ya sasa na TRA.", "correct_answer_en": "1 September 2025. Full implementation rules are still pending the Commissioner General\'s notice. Confirm current status with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_013", "subdomain": "vat_registration", "question_sw": "Je, mfanyabiashara ambaye bado hajafika kizingiti cha VAT anaweza kujisajilisha VAT kwa hiari?", "question_en": "Can a business owner below the VAT threshold voluntarily register for VAT?", "correct_answer_sw": "Ndiyo. Usajilishaji wa hiari wa VAT unaruhusiwa. Faida kuu ni uwezo wa kudai VAT ya pembejeo kwenye manunuzi ya biashara. Wasiliana na TRA kwa maelezo ya mchakato wa usajilishaji wa hiari.", "correct_answer_en": "Yes. Voluntary VAT registration is permitted. The main advantage is the ability to claim input VAT credits on business purchases. Contact TRA for details on the voluntary registration process.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_014", "subdomain": "vat_registration", "question_sw": "Je, wakili aliyesajiliwa lazima asajilishe VAT hata kama mapato yake ya mwaka ni chini ya TZS milioni 200?", "question_en": "Must a registered lawyer register for VAT even with annual revenue below TZS 200 million?", "correct_answer_sw": "Ndiyo. Wataalamu fulani \\u2014 wakili, wahasibu waliosajiliwa, wahandisi, wasanifu majengo \\u2014 wanahitajika kusajilisha VAT bila kujali mauzo yao. Hii ni sharti maalum la TRA. Thibitisha orodha kamili na TRA.", "correct_answer_en": "Yes. Certain professionals \\u2014 lawyers, registered accountants, engineers, architects \\u2014 must register for VAT regardless of revenue. This is a specific TRA requirement. Confirm the full list with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_015", "subdomain": "vat_registration", "question_sw": "Huduma za \'exempt\' (zisizo na VAT) \\u2014 je, mfanyabiashara anaweza kudai VAT ya pembejeo inayohusiana nazo?", "question_en": "For \'exempt\' VAT supplies \\u2014 can a business claim related input VAT credits?", "correct_answer_sw": "Hapana. Bidhaa/huduma za exempt hazilipishwi VAT, NA mfanyabiashara HAWEZI kudai VAT ya pembejeo inayohusiana nazo. Tofauti na zero-rated: zero-rated bado zinakuruhusu kudai VAT ya pembejeo. Thibitisha na TRA.", "correct_answer_en": "No. Exempt supplies are not subject to VAT, AND a business CANNOT claim related input VAT credits. Unlike zero-rated supplies, which still allow input VAT claims. Confirm with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_016", "subdomain": "vat_registration", "question_sw": "Je, biashara isiyosajiliwa VAT inaweza kuweka VAT ya asilimia 18 kwenye invoice zake na kukusanya pesa hiyo kutoka wateja?", "question_en": "Can an unregistered business charge 18% VAT on its invoices and collect it from customers?", "correct_answer_sw": "Hapana \\u2014 hii ni kinyume cha sheria. Ni biashara zilizosajiliwa VAT peke yake zinazoruhusiwa kukusanya VAT. Biashara isiyosajiliwa inayofanya hivi inaweza kukabili adhabu kali za TRA. Jisajilishe kwanza kama mauzo yako yamefika kizingiti.", "correct_answer_en": "No \\u2014 this is illegal. Only VAT-registered businesses may collect VAT. An unregistered business doing so faces serious TRA penalties. Register first if your sales have reached the threshold.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_017", "subdomain": "vat_registration", "question_sw": "Je, biashara inayouza bidhaa za zero-rated inaweza kudai VAT ya pembejeo kwenye gharama zake?", "question_en": "Can a business selling zero-rated goods claim input VAT on its costs?", "correct_answer_sw": "Ndiyo. Bidhaa za zero-rated bado zinachukuliwa kama \'usambazaji wa VAT\' ingawa kiwango chake ni 0%. Hivyo mfanyabiashara anaweza kudai VAT ya pembejeo kwenye manunuzi yanayohusiana. Thibitisha na TRA.", "correct_answer_en": "Yes. Zero-rated goods are still treated as \'VAT supplies\' even though the rate is 0%. So the business can claim input VAT credits on related purchases. Confirm with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_018", "subdomain": "vat_registration", "question_sw": "Je, kiwango cha VAT ya B2C ya kidijitali cha asilimia 16 kimetekelezwa kikamilifu nchini Tanzania kufikia katikati ya 2026?", "question_en": "Has the 16% B2C digital VAT rate been fully implemented in Tanzania by mid-2026?", "correct_answer_sw": "Hii haijulikani kwa uhakika bila uthibitisho wa TRA. Kiwango cha asilimia 16 kilianzishwa 1 Septemba 2025 lakini kanuni za utekelezaji bado zilisubiri tangazo la Kamishna Mkuu mwishoni mwa 2025. Thibitisha hali ya sasa moja kwa moja na TRA.", "correct_answer_en": "This cannot be confirmed without TRA verification. The 16% rate was introduced 1 September 2025 but implementation rules were still pending the Commissioner General\'s notice as of late 2025. Confirm current status directly with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_019", "subdomain": "vat_registration", "question_sw": "Biashara yangu imepata mauzo ya TZS milioni 95 katika miezi 6 \\u2014 je, nimevuka kizingiti cha VAT cha lazima?", "question_en": "My business has had TZS 95 million in sales over 6 months \\u2014 have I crossed the mandatory VAT threshold?", "correct_answer_sw": "Hapana. Kizingiti cha miezi 6 ni TZS milioni 100. Mauzo ya TZS milioni 95 bado hayajafika. Hata hivyo, fuatilia kwa makini \\u2014 TZS milioni 5 zaidi katika kipindi hicho hicho itafanya usajilishaji kuwa wa lazima. Thibitisha na TRA.", "correct_answer_en": "No. The 6-month threshold is TZS 100 million. TZS 95 million has not reached it yet. However, monitor carefully \\u2014 TZS 5 million more in the same period will make registration mandatory. Confirm with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_020", "subdomain": "vat_registration", "question_sw": "Je, mhasibu aliyeidhinishwa (CPA) lazima asajilishe VAT hata kama ana wateja wachache na mapato madogo sana?", "question_en": "Must a certified public accountant (CPA) register for VAT even with very few clients and low revenue?", "correct_answer_sw": "Ndiyo. Wahasibu waliosajiliwa wanahitajika kusajilisha VAT bila kujali kiasi cha mauzo yao \\u2014 hii ni sharti maalum la TRA kwa fani zilizoorodheshwa. Mapato madogo hayatolei msamaha katika kundi hili. Thibitisha na TRA.", "correct_answer_en": "Yes. Registered accountants must register for VAT regardless of revenue \\u2014 this is a specific TRA requirement for listed professions. Low income does not exempt members of this group. Confirm with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_021", "subdomain": "vat_registration", "question_sw": "\'Mnunuzi anayehitimu\' (qualifying buyer) katika mfumo wa VAT withholding wa Tanzania maana yake nini?", "question_en": "What does \'qualifying buyer\' mean in Tanzania\'s VAT withholding system?", "correct_answer_sw": "Mnunuzi anayehitimu ni: (a) Wizara ya Fedha; (b) taasisi ya serikali inayohifadhi mapato yake yenyewe; au (c) mtu aliyeteuliwa rasmi na Kamishna Mkuu. Mnunuzi huyu anakata sehemu ya VAT na kuipeleka TRA moja kwa moja.", "correct_answer_en": "A qualifying buyer means: (a) Ministry of Finance; (b) a government entity that retains its own revenue; or (c) a person formally designated by the Commissioner General. This buyer deducts a portion of VAT and remits it directly to TRA.", "answer_type": "definition", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_022", "subdomain": "vat_registration", "question_sw": "Tofauti kati ya bidhaa za \'zero-rated\' na \'exempt\' katika VAT ya Tanzania ni nini?", "question_en": "What is the difference between \'zero-rated\' and \'exempt\' supplies in Tanzania\'s VAT?", "correct_answer_sw": "Zero-rated: kiwango cha VAT ni 0%, bado ni \'usambazaji wa VAT\' \\u2014 unaweza kudai VAT ya pembejeo. Exempt: hazilipishwi VAT kabisa \\u2014 HUWEZI kudai VAT ya pembejeo inayohusiana. Tofauti hii inaathiri sana gharama za uendeshaji wa biashara. Thibitisha na TRA.", "correct_answer_en": "Zero-rated: VAT rate is 0%, still a \'VAT supply\' \\u2014 you CAN claim input VAT. Exempt: no VAT at all \\u2014 you CANNOT claim related input VAT. This distinction significantly affects business operating costs. Confirm with TRA.", "answer_type": "definition", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_023", "subdomain": "vat_registration", "question_sw": "\'VAT ya matokeo\' (output VAT) kwa mfanyabiashara aliyesajiliwa maana yake nini?", "question_en": "What does \'output VAT\' mean for a VAT-registered business?", "correct_answer_sw": "VAT ya matokeo ni VAT unayoikusanya kutoka wateja kwenye mauzo yako ya bidhaa au huduma zinazolipwa VAT. Ni pesa ya serikali \\u2014 unaikusanya kwa niaba ya TRA na kuwasilisha tofauti kati yake na VAT ya pembejeo.", "correct_answer_en": "Output VAT is the VAT you collect from customers on taxable sales. It is government money \\u2014 collected on behalf of TRA, with only the net difference (output minus input) remitted.", "answer_type": "definition", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_024", "subdomain": "vat_registration", "question_sw": "\'Kipindi cha miezi 12 kinachozunguka\' (rolling 12-month period) katika kizingiti cha VAT maana yake nini?", "question_en": "What does \'rolling 12-month period\' mean for the VAT threshold?", "correct_answer_sw": "Ni kipindi chochote cha miezi 12 mfululizo \\u2014 si lazima iwe mwaka wa kalenda (Januari-Desemba). Kwa mfano, Machi 2025 hadi Februari 2026 ni kipindi halali. TRA inatathimini mauzo katika kipindi chochote cha miezi 12, si mwaka wa fedha peke yake.", "correct_answer_en": "Any consecutive 12-month period \\u2014 not necessarily the calendar year (January\\u2013December). For example, March 2025 to February 2026 is a valid period. TRA assesses sales across any 12-month window, not just the fiscal year.", "answer_type": "definition", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_025", "subdomain": "vat_registration", "question_sw": "Kwa nini fani kama wakili na wahandisi zinahitajika kusajilisha VAT bila kujali mauzo yao?", "question_en": "Why are professions like lawyers and engineers required to register for VAT regardless of revenue?", "correct_answer_sw": "TRA imeweka sharti maalum kwa fani zilizoorodheshwa (wakili, wahasibu, wahandisi, wasanifu majengo) kwa sababu huduma zao zinachukuliwa kuwa za kitaaluma na za lazima kusajilishwa VAT. Hii inapunguza hatari ya kukimbia kodi katika sekta za kitaalamu zenye mapato makubwa yasiyoonekana wazi. Thibitisha orodha kamili na TRA.", "correct_answer_en": "TRA has set a specific requirement for listed professions (lawyers, accountants, engineers, architects) because their services are considered professional and inherently subject to VAT registration. This reduces tax evasion risk in high-income professional sectors. Confirm the full list with TRA.", "answer_type": "definition", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_026", "subdomain": "vat_registration", "question_sw": "Biashara yangu imezidi kizingiti cha VAT wiki hii \\u2014 hatua ya kwanza ninayopaswa kuchukua ni nini?", "question_en": "My business crossed the VAT threshold this week \\u2014 what is the first step I must take?", "correct_answer_sw": "Wasiliana na TRA mara moja na omba usajilishaji wa VAT. Usiendelee kufanya biashara bila usajilishaji \\u2014 kuchelewa ni ukiukwaji wa sheria. TRA itakupa nambari ya VAT na maelekezo ya kutoa risiti za EFD. Tembelea tra.go.tz au ofisi ya karibu.", "correct_answer_en": "Contact TRA immediately and apply for VAT registration. Do not continue trading unregistered \\u2014 delay is a legal violation. TRA will issue your VAT number and guide you on EFD receipts. Visit tra.go.tz or your nearest TRA office.", "answer_type": "procedure", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_027", "subdomain": "vat_registration", "question_sw": "Ninauza huduma za ushauri kwa wizara ya serikali \\u2014 utaratibu sahihi wa invoice ya VAT kwa mnunuzi anayehitimu ni upi?", "question_en": "I sell consultancy services to a government ministry \\u2014 what is the correct VAT invoice procedure for a qualifying buyer?", "correct_answer_sw": "1) Toa invoice yenye VAT ya asilimia 18 \\u2014 kwa mfano huduma ya TZS 1,000,000 + VAT TZS 180,000 = jumla TZS 1,180,000. 2) Wizara inakata asilimia 6 ya kiasi cha VAT tu (yaani 6% ya TZS 180,000 = TZS 10,800) na kuipeleka TRA moja kwa moja. 3) Wizara inakupa sehemu iliyobaki ya jumla (TZS 1,169,200). 4) Wizara inakutoa certificate ya VAT withholding siku ile VAT inapostahili kulipwa. 5) Bado unaweza kudai VAT yako yote ya pembejeo kwenye return yako ya kawaida. Thibitisha hesabu na TRA.", "correct_answer_en": "1) Issue an invoice with 18% VAT \\u2014 e.g. service TZS 1,000,000 + VAT TZS 180,000 = total TZS 1,180,000. 2) The Ministry withholds 6% of the VAT amount only (6% of TZS 180,000 = TZS 10,800) and remits it directly to TRA. 3) The Ministry pays you the remaining total (TZS 1,169,200). 4) The Ministry issues you a VAT withholding certificate on the day VAT becomes payable. 5) You can still claim all your input VAT normally on your return. Confirm calculations with TRA.", "answer_type": "procedure", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_028", "subdomain": "vat_registration", "question_sw": "Jinsi gani ya kuomba kujisajilisha VAT kwa hiari kabla ya kufikia kizingiti?", "question_en": "How does a business apply for voluntary VAT registration before reaching the threshold?", "correct_answer_sw": "1) Wasiliana na TRA kupitia tra.go.tz au ofisi ya karibu. 2) Wasilisha ombi la usajilishaji wa hiari pamoja na BRELA certificate, TIN, na hati za biashara. 3) TRA itakagua ombi. 4) Ukipitishwa, utapata nambari ya VAT na unaweza kutoa invoice za VAT na kudai VAT ya pembejeo. Thibitisha nyaraka zinazohitajika na TRA.", "correct_answer_en": "1) Contact TRA via tra.go.tz or nearest office. 2) Submit voluntary registration application with BRELA certificate, TIN, and business records. 3) TRA reviews your application. 4) If approved, you receive a VAT number and can issue VAT invoices and claim input VAT. Confirm required documents with TRA.", "answer_type": "procedure", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_029", "subdomain": "vat_registration", "question_sw": "VAT ya pembejeo yangu inazidi VAT ya matokeo kwa mwezi huu \\u2014 nifanye nini?", "question_en": "My input VAT exceeds my output VAT this month \\u2014 what should I do?", "correct_answer_sw": "1) Wasilisha VAT return yako kwa wakati (ifikapo siku ya 20) ukionyesha ziada ya VAT ya pembejeo. 2) Unaweza kuomba TRA irudishe ziada hiyo (VAT refund) au kuibeba mbele (carry forward). 3) TRA ina mchakato maalum wa madai ya kurudisha VAT. Wasiliana na TRA au mshauri wa kodi kwa mwongozo.", "correct_answer_en": "1) File your VAT return on time (by the 20th) showing the excess input VAT. 2) You can apply for TRA to refund the excess or carry it forward against future output VAT. 3) TRA has a specific process for VAT refund claims. Contact TRA or a tax adviser for guidance.", "answer_type": "procedure", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_030", "subdomain": "vat_registration", "question_sw": "Certificate ya VAT withholding inatolewa lini kwa msambazaji \\u2014 je, ni siku ya 20 ya mwezi unaofuata?", "question_en": "When must the VAT withholding certificate be issued to the supplier \\u2014 is it on the 20th of the following month?", "correct_answer_sw": "Hapana. Siku ya 20 ni deadline ya VAT return, si ya certificate ya withholding. Certificate ya VAT withholding lazima itolewe siku ile VAT inapostahili kulipwa \\u2014 si tarehe nyingine. Hizi ni tarehe mbili tofauti kabisa zinazohusiana na majukumu tofauti.", "correct_answer_en": "No. The 20th is the VAT return deadline, not the withholding certificate deadline. The VAT withholding certificate must be issued on the day VAT becomes payable \\u2014 not on any other date. These are two entirely different deadlines covering separate obligations.", "answer_type": "procedure", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_031", "subdomain": "vat_withholding", "question_sw": "Kiwango cha VAT withholding kwenye bidhaa chini ya Finance Act 2025 ni asilimia ngapi?", "question_en": "What is the VAT withholding rate on goods under Finance Act 2025?", "correct_answer_sw": "Asilimia 3, ikiwa na ufanisi kuanzia 1 Julai 2025. Kiwango hiki kinatumika wakati mnunuzi anayehitimu ananunua bidhaa kutoka kwa msambazaji aliyesajiliwa VAT. Thibitisha na TRA.", "correct_answer_en": "3%, effective from 1 July 2025. This rate applies when a qualifying buyer purchases goods from a VAT-registered supplier. Confirm with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_032", "subdomain": "vat_withholding", "question_sw": "Kiwango cha VAT withholding kwenye huduma chini ya Finance Act 2025 ni asilimia ngapi?", "question_en": "What is the VAT withholding rate on services under Finance Act 2025?", "correct_answer_sw": "Asilimia 6, ikiwa na ufanisi kuanzia 1 Julai 2025. Kiwango hiki kinatumika kwa huduma zote zinazolipwa VAT zinazonunuliwa na mnunuzi anayehitimu. Tofauti na bidhaa ambazo ni asilimia 3. Thibitisha na TRA.", "correct_answer_en": "6%, effective from 1 July 2025. This rate applies to all VAT-taxable services purchased by a qualifying buyer. This differs from goods which attract 3%. Confirm with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_033", "subdomain": "vat_withholding", "question_sw": "Utaratibu wa VAT withholding wa Finance Act 2025 ulianza lini?", "question_en": "When did the Finance Act 2025 VAT withholding regime take effect?", "correct_answer_sw": "Tarehe 1 Julai 2025. Kuanzia tarehe hii, wanunuzi wanaohitimu wanakata asilimia 3 (bidhaa) au asilimia 6 (huduma) kutoka malipo ya VAT na kuipeleka TRA moja kwa moja.", "correct_answer_en": "1 July 2025. From this date, qualifying buyers withhold 3% (goods) or 6% (services) from VAT payments and remit directly to TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_034", "subdomain": "vat_withholding", "question_sw": "Mnunuzi anayehitimu ananunua bidhaa za TZS milioni 5 \\u2014 kiasi gani cha VAT withholding kinakatwa?", "question_en": "A qualifying buyer purchases TZS 5 million of goods \\u2014 what VAT withholding amount is deducted?", "correct_answer_sw": "TZS 150,000 (asilimia 3 ya TZS milioni 5). Mnunuzi anakata TZS 150,000 kutoka malipo ya VAT na kuipeleka TRA. Msambazaji anapokea sehemu iliyobaki ya VAT na bado anaweza kudai VAT yake ya pembejeo. Thibitisha hesabu na TRA.", "correct_answer_en": "TZS 150,000 (3% of TZS 5 million). The qualifying buyer deducts TZS 150,000 from the VAT payment and remits it to TRA. The supplier receives the remaining VAT and can still claim input VAT. Confirm calculation with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_035", "subdomain": "vat_withholding", "question_sw": "Taasisi ya serikali inayohitimu inalipa TZS milioni 3 kwa huduma za ushauri \\u2014 kiasi gani cha VAT withholding kinakatwa?", "question_en": "A qualifying government entity pays TZS 3 million for consultancy services \\u2014 what VAT withholding amount is deducted?", "correct_answer_sw": "TZS 180,000 (asilimia 6 ya TZS milioni 3). Kiwango cha huduma ni asilimia 6 \\u2014 mara mbili ya kiwango cha bidhaa (asilimia 3). Taasisi inakata TZS 180,000 na kuipeleka TRA. Thibitisha hesabu na TRA.", "correct_answer_en": "TZS 180,000 (6% of TZS 3 million). The service rate is 6% \\u2014 double the goods rate (3%). The institution deducts TZS 180,000 and remits to TRA. Confirm calculation with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_036", "subdomain": "vat_withholding", "question_sw": "Aina tatu za wanunuzi wanaohitimu (qualifying buyers) chini ya mfumo wa VAT withholding wa Finance Act 2025 ni zipi?", "question_en": "What are the three types of qualifying buyers under the Finance Act 2025 VAT withholding system?", "correct_answer_sw": "1) Wizara ya Fedha. 2) Taasisi ya serikali inayohifadhi mapato yake yenyewe. 3) Mtu aliyeteuliwa rasmi na Kamishna Mkuu. Hizi peke yake ndizo zinazostahili kukata VAT withholding. Thibitisha orodha kamili na TRA.", "correct_answer_en": "1) Ministry of Finance. 2) A government entity that retains its own revenue. 3) A person formally designated by the Commissioner General. Only these qualify to withhold VAT. Confirm the full list with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_037", "subdomain": "vat_withholding", "question_sw": "Chini ya mfumo wa VAT withholding, ni upande gani \\u2014 mnunuzi au msambazaji \\u2014 anayepeleka VAT iliyozuiwa TRA?", "question_en": "Under the VAT withholding system, which party \\u2014 buyer or supplier \\u2014 remits the withheld VAT to TRA?", "correct_answer_sw": "Mnunuzi anayehitimu (qualifying buyer). Mnunuzi anakata sehemu ya VAT kutoka malipo na anailipa TRA moja kwa moja, bila kupitia msambazaji. Msambazaji anapokea tu sehemu iliyobaki ya VAT.", "correct_answer_en": "The qualifying buyer. The qualifying buyer deducts a portion of VAT from the payment and remits it directly to TRA, bypassing the supplier. The supplier receives only the remaining VAT portion.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_038", "subdomain": "vat_withholding", "question_sw": "Tofauti ya kiwango cha VAT withholding kati ya bidhaa (asilimia 3) na huduma (asilimia 6) ni pointi ngapi za asilimia?", "question_en": "What is the percentage-point difference between the VAT withholding rate on goods (3%) and services (6%)?", "correct_answer_sw": "Pointi 3 za asilimia (6% \\u2212 3% = 3%). Kiwango cha huduma ni mara mbili ya kiwango cha bidhaa. Hii inamaanisha msambazaji wa huduma anaathirika zaidi na withholding kuliko msambazaji wa bidhaa. Thibitisha na TRA.", "correct_answer_en": "3 percentage points (6% \\u2212 3% = 3%). The services rate is double the goods rate. This means a services supplier is more significantly affected by withholding than a goods supplier. Confirm with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_039", "subdomain": "vat_withholding", "question_sw": "Je, kiwango cha VAT withholding ni sawa kwa bidhaa na huduma?", "question_en": "Is the VAT withholding rate the same for goods and services?", "correct_answer_sw": "Hapana. Kiwango cha bidhaa ni asilimia 3 na kiwango cha huduma ni asilimia 6 \\u2014 mara mbili. Hii ni tofauti muhimu inayoathiri biashara zinazotoa huduma kwa wanunuzi wa serikali. Thibitisha na TRA.", "correct_answer_en": "No. The rate for goods is 3% and for services is 6% \\u2014 double. This is an important distinction affecting businesses supplying services to government buyers. Confirm with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_040", "subdomain": "vat_withholding", "question_sw": "Je, msambazaji anapoteza haki ya kudai VAT ya pembejeo wakati mnunuzi anayehitimu anakata VAT withholding?", "question_en": "Does a supplier lose the right to claim input VAT credits when a qualifying buyer withholds VAT?", "correct_answer_sw": "Hapana. Msambazaji bado anaweza kudai VAT yake yote ya pembejeo kwa kawaida kwenye VAT return yake. VAT withholding inathiri sehemu ya VAT ya matokeo tu \\u2014 haiathiri haki ya kudai VAT ya pembejeo. Thibitisha na TRA. Hata hivyo, unaweza kudai kiasi kilichozuiwa TU kama una certificate halali ya VAT withholding wakati wa kuwasilisha return yako. Bila certificate halali, TRA haitakubali madai yako.", "correct_answer_en": "No. The supplier can still claim all their input VAT credits normally on their VAT return. Withholding only affects a portion of the output VAT \\u2014 it does not affect the right to claim input VAT. Confirm with TRA. However, you can only claim the withheld amount if you hold a valid VAT withholding certificate at the time of filing your return. Without a valid certificate, TRA will not accept your claim.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_041", "subdomain": "vat_withholding", "question_sw": "Je, utaratibu wa VAT withholding wa Finance Act 2025 ulianza kutumika tangu 1 Januari 2025?", "question_en": "Did the Finance Act 2025 VAT withholding regime start from 1 January 2025?", "correct_answer_sw": "Hapana. Utaratibu wa VAT withholding ulianza kuanzia 1 Julai 2025, si Januari 2025. Finance Act inaanza kutumika mara nyingi tarehe 1 Julai kwa Tanzania. Thibitisha tarehe sahihi na TRA.", "correct_answer_en": "No. The VAT withholding regime started from 1 July 2025, not January 2025. Finance Acts typically take effect from 1 July in Tanzania. Confirm the exact date with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_042", "subdomain": "vat_withholding", "question_sw": "Je, certificate ya VAT withholding inatakiwa ifikishwe msambazaji ifikapo siku ya 20 ya mwezi unaofuata?", "question_en": "Must the VAT withholding certificate be delivered to the supplier by the 20th of the following month?", "correct_answer_sw": "Hapana. Certificate ya VAT withholding lazima itolewe siku ile VAT inapostahili kulipwa \\u2014 si siku ya 20. Siku ya 20 ni deadline ya VAT return, ambayo ni jukumu tofauti kabisa. Thibitisha tarehe sahihi na TRA.", "correct_answer_en": "No. The VAT withholding certificate must be issued on the day VAT becomes payable \\u2014 not on the 20th. The 20th is the VAT return deadline, which is an entirely separate obligation. Confirm the exact date with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_043", "subdomain": "vat_withholding", "question_sw": "Je, kampuni ya kawaida ya biashara (isiyo ya serikali) inaweza kuwa mnunuzi anayehitimu wa VAT withholding kiotomatiki?", "question_en": "Can a regular private business (non-government) automatically be a qualifying VAT withholding buyer?", "correct_answer_sw": "Hapana kwa kawaida. Wanunuzi wanaohitimu ni: Wizara ya Fedha, taasisi za serikali zinazohifadhi mapato yake, na watu waliochaguliwa maalum na Kamishna Mkuu. Kampuni ya kawaida ya biashara haihitimu bila uteuzi maalum wa Kamishna Mkuu. Thibitisha na TRA.", "correct_answer_en": "Not automatically. Qualifying buyers are: Ministry of Finance, government entities retaining own revenue, and persons specifically designated by the Commissioner General. A regular private business does not qualify without a specific CG designation. Confirm with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_044", "subdomain": "vat_withholding", "question_sw": "Je, kiwango cha asilimia 3 cha VAT withholding kinatumika pia kwa ununuzi wa huduma za IT kutoka kwa msambazaji aliyesajiliwa VAT?", "question_en": "Does the 3% VAT withholding rate apply to purchases of IT services from a VAT-registered supplier?", "correct_answer_sw": "Hapana. Huduma za IT ni huduma, hivyo kiwango kinachofaa ni asilimia 6 \\u2014 si asilimia 3. Kiwango cha asilimia 3 kinahusu bidhaa peke yake. Thibitisha aina ya usambazaji na TRA ili kutumia kiwango sahihi.", "correct_answer_en": "No. IT services are services, so the applicable rate is 6% \\u2014 not 3%. The 3% rate applies to goods only. Confirm the supply type with TRA to apply the correct rate.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_045", "subdomain": "vat_withholding", "question_sw": "Je, Wizara ya Fedha Tanzania ni mnunuzi anayehitimu kwa madhumuni ya VAT withholding kiotomatiki?", "question_en": "Is the Tanzania Ministry of Finance automatically a qualifying buyer for VAT withholding purposes?", "correct_answer_sw": "Ndiyo. Wizara ya Fedha ni moja ya aina tatu za wanunuzi wanaohitimu chini ya Finance Act 2025 \\u2014 haihitaji uteuzi wa ziada wa Kamishna Mkuu. Thibitisha hali hii na TRA kwa miamala yako maalum.", "correct_answer_en": "Yes. The Ministry of Finance is one of the three types of qualifying buyers under Finance Act 2025 \\u2014 no additional Commissioner General designation is required. Confirm this status with TRA for your specific transactions.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_046", "subdomain": "vat_withholding", "question_sw": "\'Wakala wa VAT withholding\' (VAT withholding agent) maana yake nini?", "question_en": "What does \'VAT withholding agent\' mean?", "correct_answer_sw": "Wakala wa VAT withholding ni mnunuzi anayehitimu aliyeidhinishwa kukata sehemu ya VAT wakati wa malipo na kuipeleka TRA moja kwa moja, bila kupitia msambazaji. Mnunuzi huyu anawakilisha TRA katika ukusanyaji wa sehemu ya VAT hiyo.", "correct_answer_en": "A VAT withholding agent is a qualifying buyer authorised to deduct a portion of VAT at the point of payment and remit it directly to TRA, bypassing the supplier. This buyer acts as TRA\'s representative in collecting that VAT portion.", "answer_type": "definition", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_047", "subdomain": "vat_withholding", "question_sw": "Tofauti kati ya tarehe ya kufungua VAT return (siku ya 20) na tarehe ya certificate ya VAT withholding ni nini?", "question_en": "What is the difference between the VAT return deadline (20th) and the VAT withholding certificate deadline?", "correct_answer_sw": "VAT return: iwasilishwe ifikapo siku ya 20 ya mwezi unaofuata. Certificate ya VAT withholding: lazima itolewe siku ile VAT inapostahili kulipwa \\u2014 si siku ya 20. Hizi ni tarehe mbili tofauti kabisa zinazohusiana na majukumu tofauti ya kisheria.", "correct_answer_en": "VAT return: filed by the 20th of the following month. VAT withholding certificate: must be issued on the day VAT becomes payable \\u2014 not the 20th. These are two entirely different deadlines covering separate legal obligations.", "answer_type": "definition", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_048", "subdomain": "vat_withholding", "question_sw": "Maana ya \'VAT withholding si VAT yote\' kwa msambazaji ni nini?", "question_en": "What does it mean that \'VAT withholding is not the full VAT\' for a supplier?", "correct_answer_sw": "Mnunuzi anayehitimu anakata sehemu ndogo ya VAT tu (asilimia 3 au 6) \\u2014 si VAT yote ya asilimia 18. Msambazaji bado ana wajibu wa VAT iliyobaki kupitia return yake ya kawaida. Kiasi kilichozuiwa kinapelekwa TRA na mnunuzi, na msambazaji anadai VAT ya pembejeo yake kwa kawaida.", "correct_answer_en": "The qualifying buyer withholds only a small portion of VAT (3% or 6%) \\u2014 not the full 18% VAT. The supplier still handles the remaining VAT through their normal return. The withheld amount is remitted to TRA by the buyer, while the supplier claims input VAT credits normally.", "answer_type": "definition", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_049", "subdomain": "vat_withholding", "question_sw": "Mnunuzi anayehitimu akishindwa kukata na kupeleka VAT withholding TRA \\u2014 matokeo yake ni nini?", "question_en": "What are the consequences if a qualifying buyer fails to withhold and remit VAT to TRA?", "correct_answer_sw": "TRA inaweza kumshika mnunuzi anayehitimu mwenyewe kuwajibika kwa VAT isiyopelekwa, pamoja na riba na faini. Kushindwa kuzuia VAT kunaweza pia kusababisha ukaguzi wa TRA na hatua za kisheria. Thibitisha majukumu kamili na TRA.", "correct_answer_en": "TRA can hold the qualifying buyer itself liable for the unremitted VAT, plus interest and penalties. Failure to withhold can also trigger a TRA audit and legal action. Confirm full obligations with TRA.", "answer_type": "penalty", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_050", "subdomain": "vat_withholding", "question_sw": "Msambazaji akidanganya kiwango cha VAT kwenye invoice kwa mnunuzi anayehitimu \\u2014 hatari ni nini?", "question_en": "If a supplier misrepresents the VAT rate on an invoice to a qualifying buyer \\u2014 what is the risk?", "correct_answer_sw": "TRA inaweza kufanya ukaguzi wa VAT, kudai VAT yote iliyopotea pamoja na riba na faini. Kudanganya kwa makusudi kwenye invoice za VAT kunaweza kuchukuliwa kama udanganyifu wa kodi. Thibitisha utaratibu sahihi wa kuandika invoice na TRA.", "correct_answer_en": "TRA can conduct a VAT audit, recover all lost VAT plus interest and penalties. Deliberate misrepresentation on VAT invoices may be treated as tax fraud. Confirm the correct invoicing procedure with TRA.", "answer_type": "penalty", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_051", "subdomain": "efd_compliance", "question_sw": "Kiwango cha chini cha thamani ya muamala ambacho kinahitaji risiti ya EFD kwa biashara zilizosajiliwa VAT ni kiasi gani?", "question_en": "What is the minimum transaction value requiring an EFD receipt for VAT-registered businesses?", "correct_answer_sw": "Hakuna kiwango cha chini. Kila muamala wa mauzo, bila kujali kiasi \\u2014 iwe TZS 100 au TZS 1,000,000 \\u2014 unahitaji risiti ya EFD. Kusema \'muamala mdogo hauhitaji EFD\' ni kosa la kisheria. Thibitisha na TRA.", "correct_answer_en": "There is no minimum. Every sales transaction, regardless of amount \\u2014 whether TZS 100 or TZS 1,000,000 \\u2014 requires an EFD receipt. Claiming \'small transactions don\'t need EFD\' is a legal error. Confirm with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_052", "subdomain": "efd_compliance", "question_sw": "Kiwango cha kawaida cha VAT kinachoonekana kwenye risiti ya EFD ya muamala wa kawaida unaolipwa VAT ni asilimia ngapi?", "question_en": "What standard VAT rate appears on an EFD receipt for a normal taxable transaction?", "correct_answer_sw": "Asilimia 18 \\u2014 kiwango cha kawaida cha VAT Tanzania. Risiti ya EFD inasajili bei bila VAT, kiasi cha VAT, na jumla inayolipwa. Thibitisha maudhui ya risiti ya EFD na TRA.", "correct_answer_en": "18% \\u2014 the standard Tanzania VAT rate. An EFD receipt records the pre-VAT price, the VAT amount, and the total payable. Confirm EFD receipt contents with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_053", "subdomain": "efd_compliance", "question_sw": "EFD machine ikiharibika \\u2014 biashara ina muda gani wa kuripoti TRA kabla ya kuendelea kufanya mauzo?", "question_en": "If an EFD machine breaks down \\u2014 how long does a business have to report to TRA before continuing sales?", "correct_answer_sw": "Biashara lazima iripoti TRA haraka iwezekanavyo \\u2014 mara tu tatizo linapotokea. Huwezi kuendelea kufanya mauzo bila EFD kabla ya kupata ruhusa ya TRA. Kuendelea bila ruhusa ni ukiukwaji wa sheria. Thibitisha utaratibu wa dharura na TRA.", "correct_answer_en": "The business must report to TRA as quickly as possible \\u2014 immediately when the problem occurs. You cannot continue making sales without an EFD before obtaining TRA authorization. Continuing without authorization is a legal violation. Confirm the emergency procedure with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_054", "subdomain": "efd_compliance", "question_sw": "VAT return inayohusiana na rekodi za EFD iwasilishwe TRA ifikapo tarehe ngapi kila mwezi?", "question_en": "By what date must the VAT return linked to EFD transaction records be filed with TRA each month?", "correct_answer_sw": "Ifikapo siku ya 20 ya mwezi unaofuata. Kwa mfano, rekodi za EFD za Machi zinawasilishwa kupitia VAT return ya Machi, inayostahili ifikapo 20 Aprili. Thibitisha na TRA.", "correct_answer_en": "By the 20th of the following month. For example, March EFD records are submitted through the March VAT return, due by 20 April. Confirm with TRA.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_055", "subdomain": "efd_compliance", "question_sw": "Kiwango gani cha VAT kinapaswa kuonekana kwenye risiti za EFD za malipo ya kidijitali ya B2C kuanzia 1 Septemba 2025?", "question_en": "What VAT rate should appear on EFD receipts for B2C digital payments from 1 September 2025?", "correct_answer_sw": "Kiwango cha B2C ya kidijitali ni asilimia 16 kuanzia 1 Septemba 2025, lakini bado linasubiri kanuni kamili za utekelezaji kutoka kwa Kamishna Mkuu. Thibitisha hali ya sasa ya utekelezaji na TRA kabla ya kubadilisha mipangilio ya EFD yako.", "correct_answer_en": "The B2C digital rate is 16% from 1 September 2025, but still pending full implementation rules from the Commissioner General. Confirm the current implementation status with TRA before changing your EFD settings.", "answer_type": "number", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_056", "subdomain": "efd_compliance", "question_sw": "Wateja wangu wanalipa pesa taslimu \\u2014 naweza kusema muamala mdogo wa taslimu chini ya TZS 10,000 hauhitaji risiti ya EFD?", "question_en": "My customers pay cash \\u2014 can I skip the EFD receipt for small cash transactions under TZS 10,000?", "correct_answer_sw": "Hapana kabisa. Aina ya malipo (taslimu, simu, benki) na kiasi cha muamala hazibadilishi wajibu wa kutoa risiti ya EFD. Kila muamala wa mauzo unahitaji risiti ya EFD. Kukosa kutoa risiti kunaweza kusababisha faini kali kutoka TRA. Thibitisha na TRA.", "correct_answer_en": "Absolutely not. Payment method (cash, mobile, bank) and transaction size do not change the EFD receipt obligation. Every sales transaction requires an EFD receipt. Failing to issue one can result in serious TRA penalties. Confirm with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_057", "subdomain": "efd_compliance", "question_sw": "Je, biashara inahitaji ruhusa ya TRA kabla ya kuendelea kufanya mauzo wakati EFD ikiwa imeharibika?", "question_en": "Does a business need TRA\'s authorization before continuing sales when an EFD is broken?", "correct_answer_sw": "Ndiyo. Lazima upate ruhusa ya TRA (kama idhini ya muda) kabla ya kuendelea kufanya mauzo kwa risiti za kawaida. Kuendelea bila ruhusa ni ukiukwaji wa sheria wa VAT. Ripoti tatizo TRA mara moja na subiri maelekezo yao.", "correct_answer_en": "Yes. You must obtain TRA authorization (such as a temporary permit) before continuing sales with manual receipts. Continuing without authorization is a VAT legal violation. Report the problem to TRA immediately and wait for their guidance.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_058", "subdomain": "efd_compliance", "question_sw": "Je, wateja wana haki ya kisheria ya kudai risiti ya EFD kutoka kwa biashara iliyosajiliwa VAT?", "question_en": "Do customers have a legal right to demand an EFD receipt from a VAT-registered business?", "correct_answer_sw": "Ndiyo. Wateja wana haki ya kudai risiti ya EFD, na TRA inawahimiza wateja kuripoti biashara ambazo hazitoai risiti. Biashara ina wajibu wa kisheria wa kutoa risiti \\u2014 si hiari.", "correct_answer_en": "Yes. Customers have the right to demand an EFD receipt, and TRA encourages customers to report businesses that do not issue receipts. Businesses have a legal obligation to issue receipts \\u2014 it is not optional.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_059", "subdomain": "efd_compliance", "question_sw": "Mteja akilipa kupitia mobile money \\u2014 je, ninaweza kutoa risiti iliyoandikwa kwa mkono badala ya risiti ya EFD?", "question_en": "A customer pays via mobile money \\u2014 can I issue a handwritten receipt instead of an EFD receipt?", "correct_answer_sw": "Hapana. Njia ya malipo haina umuhimu \\u2014 risiti ya EFD inahitajika kwa kila muamala, iwe malipo ya taslimu, mobile money, kadi, au benki. Risiti iliyoandikwa kwa mkono badala ya EFD ni ukiukwaji wa sheria ya VAT. Thibitisha na TRA.", "correct_answer_en": "No. Payment method is irrelevant \\u2014 an EFD receipt is required for every transaction, whether cash, mobile money, card, or bank. A handwritten receipt instead of EFD is a VAT law violation. Confirm with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_060", "subdomain": "efd_compliance", "question_sw": "Je, TRA inaweza kufunga biashara kwa kutotoa risiti za EFD mara kwa mara?", "question_en": "Can TRA close down a business for repeatedly failing to issue EFD receipts?", "correct_answer_sw": "Ndiyo. TRA ina mamlaka ya kutoza faini, kusimamisha leseni ya biashara, na kuchukua hatua nyingine za kisheria ikiwa ni pamoja na kufunga biashara. Kukosa kutoa risiti za EFD mara kwa mara kunaweza kuchukuliwa kama udanganyifu wa kodi. Thibitisha na TRA.", "correct_answer_en": "Yes. TRA has authority to impose fines, suspend business licences, and take other legal action including business closure. Repeatedly failing to issue EFD receipts may be treated as tax fraud. Confirm with TRA.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_061", "subdomain": "efd_compliance", "question_sw": "Je, mashine ya EFD inatuma taarifa za miamala moja kwa moja TRA bila mfanyabiashara kufanya chochote zaidi?", "question_en": "Does an EFD machine transmit transaction data directly to TRA automatically?", "correct_answer_sw": "Ndiyo. Mashine ya EFD imeunganishwa moja kwa moja na mfumo wa TRA na inasajili na kutuma taarifa za kila muamala wa mauzo. Hii inamaanisha TRA inaweza kuona rekodi zako za mauzo moja kwa moja \\u2014 hivyo kuficha miamala ni vigumu sana.", "correct_answer_en": "Yes. The EFD machine is directly linked to TRA\'s system and records and transmits data for every sales transaction. This means TRA can see your sales records directly \\u2014 making transaction concealment very difficult.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_062", "subdomain": "efd_compliance", "question_sw": "EFD ikiwa imeharibika na inasubiri ukarabati \\u2014 je, biashara inaweza kutumia risiti za kawaida bila kuomba ruhusa ya TRA?", "question_en": "If an EFD is being repaired \\u2014 can a business use manual receipts without obtaining TRA permission?", "correct_answer_sw": "Hapana. Kabla ya kutumia risiti za kawaida wakati EFD ikiwa inakarabatiwa, lazima upate ruhusa (idhini ya muda) kutoka TRA kwanza. Kutumia risiti za kawaida bila ruhusa hii ni ukiukwaji wa sheria. Ripoti TRA mara moja na subiri maelekezo yao.", "correct_answer_en": "No. Before using manual receipts while an EFD is being repaired, you must first obtain TRA\'s authorization (temporary permit). Using manual receipts without this authorization is a legal violation. Report to TRA immediately and await their guidance.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_063", "subdomain": "efd_compliance", "question_sw": "Je, ni jukumu la mteja kuomba risiti ya EFD, au biashara inapaswa kuitoa moja kwa moja bila kuombwa?", "question_en": "Is it the customer\'s responsibility to request an EFD receipt, or must the business issue it automatically?", "correct_answer_sw": "Ni jukumu la biashara kutoa risiti ya EFD mara moja baada ya kila muamala \\u2014 bila kusubiri mteja aombe. Biashara haiwezi kutegemea mteja asikuomba ili kutooa risiti. Kukosa kutoa risiti bila kuombwa ni ukiukwaji wa sheria.", "correct_answer_en": "It is the business\'s obligation to issue an EFD receipt immediately after every transaction \\u2014 without waiting for the customer to request one. A business cannot rely on the customer not asking in order to skip issuing a receipt. Failing to issue one proactively is a legal violation.", "answer_type": "yes_no", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_064", "subdomain": "efd_compliance", "question_sw": "EFD machine yangu imeharibika ghafla \\u2014 hatua sahihi ni zipi?", "question_en": "My EFD machine has suddenly broken down \\u2014 what are the correct steps?", "correct_answer_sw": "1) Simamisha mauzo mara moja. 2) Ripoti tatizo TRA haraka iwezekanavyo. 3) Wasiliana na msambazaji wa EFD aliyeidhinishwa na TRA kuhusu ukarabati. 4) Omba TRA idhini ya muda ya kutumia risiti za kawaida wakati wa ukarabati. 5) Usiendelee kufanya mauzo hadi utakapopata maelekezo ya TRA.", "correct_answer_en": "1) Stop sales immediately. 2) Report the problem to TRA as quickly as possible. 3) Contact your TRA-approved EFD supplier about repairs. 4) Request TRA\'s temporary authorization to use manual receipts during repairs. 5) Do not resume sales until you receive TRA guidance.", "answer_type": "procedure", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_065", "subdomain": "efd_compliance", "question_sw": "Biashara mpya iliyosajiliwa VAT inapata EFD machine vipi?", "question_en": "How does a newly VAT-registered business obtain an EFD machine?", "correct_answer_sw": "1) Wasiliana na TRA kwa orodha ya wasambazaji wa EFD walioidhinishwa Tanzania. 2) Nunua au kodi EFD kutoka kwa msambazaji aliyeidhinishwa \\u2014 usiuchukue mwingine yeyote. 3) Msambazaji atafunga EFD na kuiandikisha kwenye mfumo wa TRA. 4) Baada ya ufungaji, EFD itaunganishwa moja kwa moja na TRA. Thibitisha gharama na msambazaji.", "correct_answer_en": "1) Contact TRA for the list of TRA-approved EFD suppliers in Tanzania. 2) Purchase or lease an EFD from an approved supplier only. 3) The supplier installs and registers the EFD on TRA\'s system. 4) After installation, the EFD is directly linked to TRA. Confirm costs with the supplier.", "answer_type": "procedure", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_066", "subdomain": "efd_compliance", "question_sw": "Mteja anakataa kukubali risiti ya EFD \\u2014 nifanye nini?", "question_en": "A customer refuses to accept the EFD receipt \\u2014 what should I do?", "correct_answer_sw": "Toa risiti ya EFD hata kama mteja anakataa kukubali. Wajibu wa kisheria ni wako wewe kama mfanyabiashara \\u2014 si mteja. Rekodi ya EFD inasajiliwa kwenye mfumo wa TRA iwe mteja alichukua risiti au la. Huwezi kufuta muamala au kukosa kutoa risiti kwa sababu mteja alikataa.", "correct_answer_en": "Issue the EFD receipt even if the customer refuses to accept it. The legal obligation is yours as the business owner \\u2014 not the customer\'s. The EFD record is registered in TRA\'s system whether or not the customer took the receipt. You cannot cancel the transaction or skip the receipt because the customer declined.", "answer_type": "procedure", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_067", "subdomain": "efd_compliance", "question_sw": "TRA kawaida hugundua vipi biashara ambazo hazitoi risiti za EFD?", "question_en": "How does TRA typically identify businesses that are not issuing EFD receipts?", "correct_answer_sw": "TRA hutumia mbinu nne kuu: (1) ukaguzi wa ghafla wa maeneo ya biashara, (2) malalamiko ya wateja, (3) uchambuzi wa data wa EFD dhidi ya benki na mobile money, (4) tofauti kati ya mauzo yaliyoripotiwa na miamala halisi. Mbinu moja tu inatosha kufungua ukaguzi kamili. Thibitisha utaratibu wako wa EFD daima.", "correct_answer_en": "TRA uses four main methods: (1) surprise inspections of business premises, (2) customer complaints, (3) EFD data analysis cross-referenced against bank and mobile money records, (4) discrepancies between reported sales and actual transactions. Any one method alone can trigger a full audit. Always maintain EFD compliance.", "answer_type": "procedure", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_068", "subdomain": "efd_compliance", "question_sw": "Adhabu za TRA kwa biashara inayokataa kutoa risiti za EFD mara kwa mara ni zipi?", "question_en": "What penalties can TRA impose on a business that repeatedly refuses to issue EFD receipts?", "correct_answer_sw": "TRA inaweza: kutoza faini kubwa, kusimamisha leseni ya biashara, kufunga biashara, na kuchukulia miamala isiyokuwa na risiti kama udanganyifu wa kodi. Katika hali mbaya, wakurugenzi wa kampuni wanaweza kuwajibika kisheria. Thibitisha kiasi cha faini na TRA.", "correct_answer_en": "TRA can: impose large fines, suspend business licences, close the business, and treat transactions without receipts as tax fraud. In serious cases, company directors can be held personally liable. Confirm current penalty amounts with TRA.", "answer_type": "penalty", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_069", "subdomain": "efd_compliance", "question_sw": "Mteja akiripoti biashara kwa TRA kwa kutotoa risiti ya EFD \\u2014 TRA inaweza kufanya nini?", "question_en": "If a customer reports a business to TRA for not issuing an EFD receipt \\u2014 what can TRA do?", "correct_answer_sw": "TRA inaweza kufanya ukaguzi wa ghafla wa biashara hiyo, kutoza faini, na kufanya ukaguzi kamili wa rekodi za mauzo. Ripoti moja inaweza kuchochea ukaguzi wa kina ambao unagundua miamala mingi isiyoandikwa. Hii ni hatari kubwa kwa biashara yoyote inayokimbia EFD.", "correct_answer_en": "TRA can conduct a surprise inspection of that business, impose fines, and carry out a full audit of sales records. A single report can trigger a thorough audit that uncovers many unrecorded transactions. This is a significant risk for any business avoiding EFD compliance.", "answer_type": "penalty", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_070", "subdomain": "efd_compliance", "question_sw": "Je, kukosa kutoa risiti ya EFD kunaweza kuchukuliwa kama udanganyifu wa kodi (tax evasion) na TRA?", "question_en": "Can failing to issue an EFD receipt be treated as tax evasion by TRA?", "correct_answer_sw": "Ndiyo. Miamala isiyokuwa na risiti za EFD inaweza kutafsiriwa na TRA kama jaribio la kuficha mauzo na kukimbia kulipa VAT. Hii inaweza kusababisha madai ya udanganyifu wa kodi, ambayo yana adhabu kali zaidi kuliko faini za kawaida. Toa risiti ya EFD kwa kila muamala daima.", "correct_answer_en": "Yes. Transactions without EFD receipts can be interpreted by TRA as an attempt to conceal sales and avoid paying VAT. This can lead to tax evasion charges, which carry more severe penalties than standard fines. Always issue an EFD receipt for every transaction.", "answer_type": "penalty", "source_url": "https://kpmg.com/tz/en/home/insights/2025/07/tanzania-finance-act-2025.html"}, {"id": "eval_071", "subdomain": "brela_registration", "question_sw": "BRELA inasajilisha aina kuu ngapi za miundo ya biashara Tanzania?", "question_en": "How many main types of business structure does BRELA register in Tanzania?", "correct_answer_sw": "Tatu: (1) Jina la biashara (business name / sole trader), (2) Ushirikiano (partnership), na (3) Kampuni yenye ukomo (limited liability company). Aina nyingine kama kampuni za umma pia zinawezekanwa. Thibitisha na BRELA kwenye brela.go.tz.", "correct_answer_en": "Three: (1) Business name / sole trader, (2) Partnership, and (3) Limited liability company. Other types such as public companies are also possible. Confirm with BRELA at brela.go.tz.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_072", "subdomain": "brela_registration", "question_sw": "Mara ngapi kampuni iliyosajiliwa na BRELA inahitajika kuwasilisha annual return (ripoti ya kila mwaka)?", "question_en": "How often must a BRELA-registered company file an annual return?", "correct_answer_sw": "Mara moja kwa mwaka. Annual return inaweka rekodi za BRELA za hivi karibuni \\u2014 ikiwa ni pamoja na majina ya wakurugenzi, anwani ya ofisi, na maelezo ya wamiliki wa hisa. Kushindwa kuwasilisha kunaweza kusababisha faini au kufutwa kwa usajilishaji. Thibitisha tarehe ya kufunga na BRELA.", "correct_answer_en": "Once per year. The annual return keeps BRELA\'s records current \\u2014 including director names, registered office address, and shareholder details. Failure to file can result in fines or deregistration. Confirm the filing deadline with BRELA.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_073", "subdomain": "brela_registration", "question_sw": "Jina la biashara (sole trader) linalosajiliwa BRELA linaweza kumilikiwa na watu wangapi?", "question_en": "How many people can own a business name (sole trader) registered with BRELA?", "correct_answer_sw": "Mtu mmoja tu. Jina la biashara (business name) ni kwa mfanyabiashara mmoja peke yake (sole trader). Ukitaka kumiliki biashara pamoja na mtu mwingine, unahitaji ushirikiano (partnership) au kampuni yenye ukomo (limited company). Thibitisha na BRELA.", "correct_answer_en": "Only one person. A business name is for a single sole trader only. If you want to co-own a business with another person, you need a partnership or a limited company. Confirm with BRELA.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_074", "subdomain": "brela_registration", "question_sw": "Mfanyabiashara anahitaji nyaraka kuu ngapi kutoka taasisi tofauti (BRELA na TRA) ili kufanya biashara rasmi Tanzania?", "question_en": "How many key documents from different institutions (BRELA and TRA) does a business owner need to operate formally in Tanzania?", "correct_answer_sw": "Angalau mbili: (1) Certificate ya BRELA \\u2014 inathibitisha usajilishaji wa kisheria wa biashara; (2) TIN kutoka TRA \\u2014 nambari ya mlipakodi. Hizi mbili ni za lazima na ni tofauti \\u2014 BRELA inasajilisha biashara yenyewe; TRA inaisajilisha kwa madhumuni ya kodi. Usisajilishe moja na uache nyingine.", "correct_answer_en": "At least two: (1) BRELA certificate \\u2014 confirms legal business registration; (2) TIN from TRA \\u2014 tax identification number. Both are mandatory and distinct \\u2014 BRELA registers the business entity; TRA registers it for tax purposes. Do not register with one and skip the other.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_075", "subdomain": "brela_registration", "question_sw": "Je, mfanyabiashara anayefanya kazi peke yake kwa jina lake la kawaida bado anahitaji kusajilisha jina la biashara na BRELA?", "question_en": "Does a sole trader operating under their own personal name still need to register a business name with BRELA?", "correct_answer_sw": "Ndiyo. Sheria ya Tanzania inahitaji usajili rasmi wa biashara yoyote inayofanya shughuli za kibiashara, hata kama mfanyabiashara anatumia jina lake mwenyewe. Kutosajilisha kunaleta hatari za kisheria. Tembelea brela.go.tz kwa maelezo ya usajili wa jina la biashara.", "correct_answer_en": "Yes. Tanzanian law requires formal business registration for anyone conducting commercial activities, even when trading under their own name. Failure to register creates legal risk. Visit brela.go.tz for sole trader registration details.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_076", "subdomain": "brela_registration", "question_sw": "Je, certificate ya BRELA peke yake inatosha kufanya biashara kikamilifu bila kupata TIN kutoka TRA?", "question_en": "Is a BRELA certificate alone sufficient to operate as a fully compliant business without a TIN from TRA?", "correct_answer_sw": "Hapana. Certificate ya BRELA na TIN kutoka TRA ni nyaraka mbili tofauti zinazotakiwa. BRELA inasajilisha biashara yenyewe; TRA inaisajilisha kwa madhumuni ya kodi. Bila TIN, huwezi kulipa kodi, kuomba tender za serikali, au kufungua akaunti za biashara nyingi za benki. Pata zote mbili.", "correct_answer_en": "No. A BRELA certificate and a TIN from TRA are two separate required documents. BRELA registers the business entity; TRA registers it for tax purposes. Without a TIN, you cannot pay taxes, apply for government tenders, or open many business bank accounts. Obtain both.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_077", "subdomain": "brela_registration", "question_sw": "Je, mfanyabiashara anaweza kutafuta jina la biashara kwenye tovuti ya BRELA kabla ya kusajilisha ili kujua kama linachukuliwa tayari?", "question_en": "Can a business owner search for a business name on BRELA\'s website before registering to check if it\'s already taken?", "correct_answer_sw": "Ndiyo. BRELA inakuruhusu kufanya utafutaji wa jina la biashara kwenye brela.go.tz kabla ya kusajilisha. Baada ya kuthibitisha jina linapatikana, unaweza kuomba kulihifadhi (reservation) kabla ya kukamilisha usajilishaji wako. Jibu la utafutaji si dhamana \\u2014 thibitisha na BRELA moja kwa moja.", "correct_answer_en": "Yes. BRELA allows you to search for a business name on brela.go.tz before registering. After confirming the name is available, you can apply to reserve it before completing registration. A search result is not a guarantee \\u2014 confirm directly with BRELA.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_078", "subdomain": "brela_registration", "question_sw": "Je, kampuni inaweza kufutwa usajilishaji wake BRELA kwa kushindwa kuwasilisha annual return?", "question_en": "Can a company\'s BRELA registration be cancelled for failing to file annual returns?", "correct_answer_sw": "Ndiyo. Kushindwa kuwasilisha annual return kwa wakati kunaweza kusababisha faini na hatimaye BRELA kufuta usajilishaji wa kampuni hiyo. Kampuni iliyofutwa usajilishaji wake haiwezi kufanya biashara rasmi. Hakikisha unawasilisha annual return kila mwaka kwa wakati. Thibitisha tarehe ya kufunga na BRELA.", "correct_answer_en": "Yes. Failure to file annual returns on time can result in fines and ultimately BRELA cancelling the company\'s registration. A deregistered company cannot conduct formal business. Ensure you file annual returns every year on time. Confirm the filing deadline with BRELA.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_079", "subdomain": "brela_registration", "question_sw": "Je, ushirikiano (partnership) wa watu wawili wanaomiliki duka pamoja unahitaji kusajiliwa BRELA?", "question_en": "Does a partnership of two people co-owning a shop need to be registered with BRELA?", "correct_answer_sw": "Ndiyo. Ushirikiano wa biashara unahitajika kusajiliwa na BRELA chini ya Companies Act na sheria za Tanzania. Ushirikiano usiosajiliwa unakabiliwa na hatari za kisheria na unaweza kuathiri mgawanyo wa mali iwapo kutoelewana kutatokea. Thibitisha na BRELA kwenye brela.go.tz.", "correct_answer_en": "Yes. A business partnership must be registered with BRELA under Tanzania\'s laws. An unregistered partnership faces legal risks and can complicate asset division if disputes arise. Confirm with BRELA at brela.go.tz.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_080", "subdomain": "brela_registration", "question_sw": "BRELA ni nini na kazi yake kuu ni nini?", "question_en": "What is BRELA and what is its primary function?", "correct_answer_sw": "BRELA ni Business Registrations and Licensing Agency \\u2014 shirika la serikali la Tanzania linalosimamia usajilishaji wa biashara, ikiwa ni pamoja na makampuni, majina ya biashara, ushirikiano, na haki za miliki ya akili (intellectual property). Kila biashara inayotaka kufanya kazi rasmi Tanzania lazima isajiliwe kupitia BRELA au mamlaka inayohusika.", "correct_answer_en": "BRELA is the Business Registrations and Licensing Agency \\u2014 a Tanzanian government body that oversees business registration, including companies, business names, partnerships, and intellectual property rights. Every business wishing to operate formally in Tanzania must be registered through BRELA or the relevant authority.", "answer_type": "definition", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_081", "subdomain": "brela_registration", "question_sw": "Tofauti kuu kati ya jina la biashara (sole trader) na kampuni yenye ukomo (limited company) kuhusiana na wajibu wa kisheria ni nini?", "question_en": "What is the key legal liability difference between a business name (sole trader) and a limited company?", "correct_answer_sw": "Sole trader: wewe na biashara ni mtu mmoja kisheria \\u2014 madeni ya biashara ni madeni yako binafsi, na mali yako binafsi inaweza kuchukuliwa. Limited company: kampuni ni mtu wa kisheria tofauti nawe \\u2014 kwa kawaida mali yako binafsi inalindwa dhidi ya madeni ya kampuni. Limited company inahitaji uwasilishaji wa hesabu na gharama zaidi za usimamizi.", "correct_answer_en": "Sole trader: you and the business are the same legal person \\u2014 business debts are your personal debts and your personal assets can be seized. Limited company: the company is a separate legal entity \\u2014 your personal assets are generally protected from the company\'s debts. A limited company requires annual financial reporting and higher management costs.", "answer_type": "definition", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_082", "subdomain": "brela_registration", "question_sw": "Annual return ya BRELA (ripoti ya kila mwaka) ni nini?", "question_en": "What is a BRELA annual return?", "correct_answer_sw": "Annual return ni hati inayowasilishwa kwa BRELA kila mwaka ili kusasisha rekodi za kampuni \\u2014 ikiwa ni pamoja na majina ya wakurugenzi wa sasa, anwani ya ofisi ya makao makuu, na maelezo ya wamiliki wa hisa. Inafanya rekodi za umma za kampuni ziwe za hivi karibuni na sahihi. Thibitisha tarehe ya kufunga na BRELA.", "correct_answer_en": "An annual return is a document filed with BRELA each year to update company records \\u2014 including current director names, registered office address, and shareholder details. It keeps the company\'s public records current and accurate. Confirm the filing deadline with BRELA.", "answer_type": "definition", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_083", "subdomain": "brela_registration", "question_sw": "Usajilishaji wa miliki ya akili (intellectual property) nchini Tanzania hufanywa na taasisi gani?", "question_en": "Which body in Tanzania handles intellectual property registration?", "correct_answer_sw": "BRELA inasimamia usajilishaji wa alama za biashara (trademarks), hati miliki (patents), na miundo ya viwanda (industrial designs). Haki za ubunifu (copyrights) zinasimamiwa na COSOTA \\u2014 Copyright Society of Tanzania, si BRELA. Thibitisha aina ya ulinzi unaohitajika na taasisi husika.", "correct_answer_en": "BRELA handles registration of trademarks, patents, and industrial designs. Copyrights are handled by COSOTA \\u2014 the Copyright Society of Tanzania, not BRELA. Confirm the type of protection you need with the relevant institution.", "answer_type": "definition", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_084", "subdomain": "brela_registration", "question_sw": "Hatua za kusajilisha jina la biashara (sole trader) na BRELA ni zipi?", "question_en": "What are the steps to register a business name (sole trader) with BRELA?", "correct_answer_sw": "1) Tafuta upatikanaji wa jina kwenye brela.go.tz. 2) Omba kulihifadhi (reservation) kama linapatikana. 3) Wasilisha fomu ya usajilishaji pamoja na vitambulisho vya kitaifa na nyaraka zinazohitajika. 4) Lipa ada ya usajilishaji. 5) Pata certificate ya BRELA. Thibitisha nyaraka na ada za sasa na BRELA moja kwa moja.", "correct_answer_en": "1) Search for name availability at brela.go.tz. 2) Apply for reservation if available. 3) Submit the registration form with national ID and required documents. 4) Pay the registration fee. 5) Receive BRELA certificate. Confirm current required documents and fees directly with BRELA.", "answer_type": "procedure", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_085", "subdomain": "brela_registration", "question_sw": "Kampuni ilishindwa kuwasilisha annual return kwa miaka 2 \\u2014 matokeo yanayoweza kutokea ni yapi?", "question_en": "A company has failed to file its annual return for 2 years \\u2014 what are the potential consequences?", "correct_answer_sw": "Matokeo yanayoweza kutokea: faini kubwa kwa kila mwaka uliokosekana; uwezekano wa BRELA kufuta usajilishaji wa kampuni; kutoweza kuwasilisha hati nyingine zinazohusiana na BRELA; vikwazo vya kibiashara kama kukosa uwezo wa kufungua akaunti za benki au kushiriki kwenye zabuni. Wasiliana na BRELA mara moja ili kushughulikia tatizo hili.", "correct_answer_en": "Potential consequences: substantial fines for each missed year; possible BRELA deregistration of the company; inability to file other BRELA-related documents; commercial restrictions such as inability to open bank accounts or participate in tenders. Contact BRELA immediately to address this issue.", "answer_type": "procedure", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_086", "subdomain": "nssf_contributions", "question_sw": "Kiwango cha mchango wa NSSF kwa upande wa mwajiri ni asilimia ngapi ya mshahara wa jumla wa mfanyakazi?", "question_en": "What is the NSSF contribution rate for the employer as a percentage of an employee\'s gross salary?", "correct_answer_sw": "Asilimia 10 ya mshahara wa jumla (gross salary) wa mfanyakazi. Mwajiri analipa sehemu hii kutoka pesa zake mwenyewe \\u2014 haikatwi kutoka mshahara wa mfanyakazi. Thibitisha kiwango cha sasa na NSSF kwenye nssf.or.tz.", "correct_answer_en": "10% of the employee\'s gross salary. The employer pays this from their own funds \\u2014 it is not deducted from the employee\'s salary. Confirm the current rate with NSSF at nssf.or.tz.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_087", "subdomain": "nssf_contributions", "question_sw": "Kiwango cha mchango wa NSSF kwa upande wa mfanyakazi ni asilimia ngapi ya mshahara wake wa jumla?", "question_en": "What is the NSSF contribution rate for the employee as a percentage of their gross salary?", "correct_answer_sw": "Asilimia 10 ya mshahara wake wa jumla (gross salary). Mwajiri anakata kiasi hiki kutoka mshahara wa mfanyakazi kabla ya kulipa mshahara, kisha anachanganya na sehemu yake ya asilimia 10 na kupeleka NSSF. Thibitisha kiwango na NSSF kwenye nssf.or.tz.", "correct_answer_en": "10% of the employee\'s gross salary. The employer deducts this from the employee\'s salary before payment, then combines it with their own 10% contribution and remits to NSSF. Confirm the rate with NSSF at nssf.or.tz.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_088", "subdomain": "nssf_contributions", "question_sw": "Jumla ya mchango wa NSSF (mwajiri + mfanyakazi) ni asilimia ngapi ya mshahara wa jumla?", "question_en": "What is the total NSSF contribution rate (employer + employee combined) as a percentage of gross salary?", "correct_answer_sw": "Asilimia 20 ya mshahara wa jumla \\u2014 asilimia 10 kutoka mwajiri na asilimia 10 kutoka mfanyakazi. Jumla hii inapelekwa NSSF kila mwezi. Thibitisha na nssf.or.tz.", "correct_answer_en": "20% of gross salary \\u2014 10% from the employer and 10% from the employee. This total is remitted to NSSF every month. Confirm at nssf.or.tz.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_089", "subdomain": "nssf_contributions", "question_sw": "Deadline ya kupeleka michango ya NSSF kwa mwezi mmoja ni tarehe ngapi ya mwezi unaofuata?", "question_en": "By what date of the following month must monthly NSSF contributions be remitted?", "correct_answer_sw": "Kwa mujibu wa Sheria ya NSSF, michango inalipwa ndani ya mwezi mmoja baada ya kulipa mishahara. Kwa vitendo, NSSF karibu inatarajia malipo ifikapo siku ya 10 ya mwezi unaofuata. Thibitisha tarehe sahihi na nssf.or.tz.", "correct_answer_en": "Under the NSSF Act, contributions must be paid within one month of salary payment. In practice, NSSF generally expects payment by the 10th of the following month. Confirm the exact date at nssf.or.tz.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_090", "subdomain": "nssf_contributions", "question_sw": "Mfanyakazi ana mshahara wa jumla wa TZS 500,000 kwa mwezi \\u2014 mwajiri anachangia kiasi gani NSSF kwa sehemu yake?", "question_en": "An employee has a gross monthly salary of TZS 500,000 \\u2014 how much does the employer contribute to NSSF for their share?", "correct_answer_sw": "TZS 50,000 (asilimia 10 ya TZS 500,000). Hii ni sehemu ya mwajiri peke yake. Mfanyakazi pia anachangia TZS 50,000 (asilimia 10 yake), hivyo jumla inayopelekwa NSSF ni TZS 100,000 kwa mwezi huo. Thibitisha hesabu na NSSF.", "correct_answer_en": "TZS 50,000 (10% of TZS 500,000). This is the employer\'s share only. The employee also contributes TZS 50,000 (their 10%), so the total remitted to NSSF is TZS 100,000 for that month. Confirm calculation with NSSF.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_091", "subdomain": "nssf_contributions", "question_sw": "Mfanyakazi anapata mshahara wa jumla wa TZS 800,000 \\u2014 kiasi gani kinakatwa mshahara wake kwa ajili ya NSSF?", "question_en": "An employee earns a gross salary of TZS 800,000 \\u2014 how much is deducted from their salary for NSSF?", "correct_answer_sw": "TZS 80,000 (asilimia 10 ya TZS 800,000). Mwajiri anakata TZS 80,000 kutoka mshahara wa mfanyakazi kabla ya kulipa \\u2014 mshahara wa mkono (net salary) utakuwa TZS 720,000 (kabla ya makato mengine kama PAYE). Thibitisha hesabu na NSSF.", "correct_answer_en": "TZS 80,000 (10% of TZS 800,000). The employer deducts TZS 80,000 from the employee\'s salary before payment \\u2014 the net salary will be TZS 720,000 (before other deductions like PAYE). Confirm calculation with NSSF.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_092", "subdomain": "nssf_contributions", "question_sw": "Wafanyakazi 5 kila mmoja anapata TZS 400,000 kwa mwezi \\u2014 mwajiri anachangia kiasi gani NSSF kwa sehemu yake peke yake?", "question_en": "5 employees each earn TZS 400,000 per month \\u2014 how much does the employer contribute to NSSF for their share alone?", "correct_answer_sw": "TZS 200,000 (5 \\u00d7 TZS 40,000). Kila mfanyakazi analazimisha mchango wa mwajiri wa TZS 40,000 (asilimia 10 ya 400,000). Jumla inayopelekwa NSSF (ikiwa ni pamoja na sehemu ya wafanyakazi) itakuwa TZS 400,000 (TZS 200,000 \\u00d7 2). Thibitisha hesabu na NSSF.", "correct_answer_en": "TZS 200,000 (5 \\u00d7 TZS 40,000). Each employee requires an employer contribution of TZS 40,000 (10% of 400,000). The total remitted to NSSF (including employee shares) will be TZS 400,000 (TZS 200,000 \\u00d7 2). Confirm calculation with NSSF.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_093", "subdomain": "nssf_contributions", "question_sw": "Mchango wa NSSF wa Januari unapaswa kupelekwa NSSF ifikapo tarehe ngapi?", "question_en": "By what date must January\'s NSSF contributions be remitted to NSSF?", "correct_answer_sw": "Kwa mujibu wa Sheria ya NSSF, michango ya Januari inalipwa ndani ya mwezi mmoja baada ya kulipa mishahara \\u2014 kwa vitendo ifikapo siku ya 10 ya Februari. Kumbuka: siku ya 20 ni deadline ya VAT return, si ya NSSF. Thibitisha tarehe sahihi na nssf.or.tz.", "correct_answer_en": "Under the NSSF Act, January contributions must be paid within one month of salary payment \\u2014 in practice by the 10th of February. Note: the 20th is the VAT return deadline, not the NSSF deadline. Confirm the exact date at nssf.or.tz.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_094", "subdomain": "nssf_contributions", "question_sw": "Mwajiri akikosa kulipa NSSF \\u2014 ni taasisi gani (TRA au NSSF) inayofuatilia na kutoza adhabu?", "question_en": "If an employer fails to pay NSSF \\u2014 which institution (TRA or NSSF) pursues and imposes penalties?", "correct_answer_sw": "NSSF \\u2014 si TRA. NSSF ni taasisi tofauti na TRA kabisa, ingawa zote mbili zinahusiana na mwajiri. NSSF ina mamlaka yake ya kisheria ya kutoza riba na faini na hata kufungua kesi za kisheria dhidi ya waajiri wanaokiuka. Thibitisha utaratibu wa makosa na NSSF kwenye nssf.or.tz.", "correct_answer_en": "NSSF \\u2014 not TRA. NSSF is a completely separate institution from TRA, though both have employer obligations. NSSF has its own legal authority to charge interest and penalties and even take legal action against non-compliant employers. Confirm the penalties procedure with NSSF at nssf.or.tz.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_095", "subdomain": "nssf_contributions", "question_sw": "Mfanyabiashara anayejitegemea (self-employed) anapojiunga NSSF kwa hiari \\u2014 analipa asilimia ngapi ya mchango wake wote (sehemu zote mbili)?", "question_en": "When a self-employed person joins NSSF voluntarily \\u2014 what total percentage do they pay (both portions)?", "correct_answer_sw": "Asilimia 20 ya mapato yao (10% ya sehemu ya mwajiri + 10% ya sehemu ya mfanyakazi). Kama mwanachama wa hiari, wewe unalipa sehemu zote mbili peke yako. Kiasi cha msingi cha mapato kinachodhibitiwa kinaweza kutofautiana \\u2014 thibitisha na NSSF kwenye nssf.or.tz.", "correct_answer_en": "20% of their income (10% employer portion + 10% employee portion). As a voluntary member, you pay both portions yourself. The base income amount used may vary \\u2014 confirm with NSSF at nssf.or.tz.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_096", "subdomain": "nssf_contributions", "question_sw": "Je, ni kosa la kisheria kwa mwajiri kutosajilisha wafanyakazi wake NSSF?", "question_en": "Is it a legal offence for an employer to not register their employees with NSSF?", "correct_answer_sw": "Ndiyo. Mwajiri yeyote anayeajiri wafanyakazi Tanzania ana wajibu wa kisheria wa kuwasajilisha NSSF. Kutosajilisha ni ukiukwaji wa Sheria ya NSSF na kunaweza kusababisha adhabu za kisheria. Thibitisha mahitaji yako maalum na NSSF kwenye nssf.or.tz.", "correct_answer_en": "Yes. Any employer hiring workers in Tanzania has a legal obligation to register them with NSSF. Failure to register is a violation of the NSSF Act and can result in legal penalties. Confirm your specific requirements with NSSF at nssf.or.tz.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_097", "subdomain": "nssf_contributions", "question_sw": "Je, wafanyakazi wa nyumbani (domestic workers) kama wasichana wa nyumba wanaostahili NSSF?", "question_en": "Do domestic workers such as housemaids qualify for NSSF coverage?", "correct_answer_sw": "Ndiyo, kwa mujibu wa Sheria ya NSSF Tanzania, wafanyakazi wa nyumbani (domestic workers) wanastahili ufikiwa wa NSSF. Mwajiri wao ana wajibu wa kuchangia NSSF kwa niaba yao. Thibitisha mahitaji ya sasa na NSSF kwenye nssf.or.tz.", "correct_answer_en": "Yes, under Tanzania\'s NSSF Act, domestic workers are covered by NSSF. Their employer has an obligation to contribute to NSSF on their behalf. Confirm current requirements with NSSF at nssf.or.tz.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_098", "subdomain": "nssf_contributions", "question_sw": "Je, mfanyabiashara anayejitegemea (self-employed) anaweza kujiunga NSSF kwa hiari bila mwajiri?", "question_en": "Can a self-employed person join NSSF voluntarily without an employer?", "correct_answer_sw": "Ndiyo. NSSF Tanzania inakubali uanachama wa hiari kwa wajitegemea na wale wasio na mwajiri rasmi. Mwanachama wa hiari anachangia peke yake na hupata haki ya mafao wakati wa kustaafu, ulemavu, au hali nyingine zinazohitimu. Thibitisha taratibu za usajilishaji na NSSF kwenye nssf.or.tz.", "correct_answer_en": "Yes. NSSF Tanzania accepts voluntary membership for the self-employed and those without a formal employer. A voluntary member contributes independently and earns entitlement to benefits at retirement, disability, or other qualifying circumstances. Confirm registration procedures with NSSF at nssf.or.tz.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_099", "subdomain": "nssf_contributions", "question_sw": "Je, mchango wa NSSF wa mwajiri (asilimia 10) unakatwa kutoka mshahara wa mfanyakazi?", "question_en": "Is the employer\'s NSSF contribution (10%) deducted from the employee\'s salary?", "correct_answer_sw": "Hapana. Sehemu ya mwajiri (asilimia 10) inalipwa na mwajiri kutoka pesa zake mwenyewe \\u2014 si kukatwa mshahara wa mfanyakazi. Sehemu inayokatwa mshahara wa mfanyakazi ni asilimia 10 yake mwenyewe. Jumla inayopelekwa NSSF ni asilimia 20, lakini asilimia 10 tu inakatwa mshahara wa mfanyakazi.", "correct_answer_en": "No. The employer\'s 10% share is paid by the employer from their own funds \\u2014 it is not deducted from the employee\'s salary. What is deducted from the employee\'s salary is only the employee\'s own 10% share. The total remitted to NSSF is 20%, but only 10% comes from salary deduction.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_100", "subdomain": "nssf_contributions", "question_sw": "Mfanyakazi anapata mshahara wa jumla wa TZS milioni 2 kwa mwezi \\u2014 je, NSSF inahusika na mshahara wote?", "question_en": "An employee earns a gross salary of TZS 2 million per month \\u2014 does NSSF apply to the full TZS 2 million?", "correct_answer_sw": "Ndiyo. NSSF inahusika na mshahara wote wa jumla bila kiwango cha juu. Mfanyakazi anachangia TZS 200,000 (10% ya TZS milioni 2) na mwajiri anachangia TZS 200,000 nyingine \\u2014 jumla ya TZS 400,000 inapelekwa NSSF kila mwezi. Thibitisha na nssf.or.tz.", "correct_answer_en": "Yes. NSSF applies to the full gross salary with no upper cap. The employee contributes TZS 200,000 (10% of TZS 2M) and the employer contributes another TZS 200,000 \\u2014 a total of TZS 400,000 remitted to NSSF each month. Confirm at nssf.or.tz.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_101", "subdomain": "nssf_contributions", "question_sw": "Je, NSSF inatoa mafao ya uzazi kwa wanawake wanaohitimu?", "question_en": "Does NSSF provide maternity benefits for qualifying female members?", "correct_answer_sw": "Ndiyo. NSSF inatoa mafao ya uzazi kwa wanawake wanaochangia wanaohitimu. Mafao mengine ni pensheni ya uzee, ulemavu, wasio na wazazi, na msaada wa mazishi. Thibitisha masharti ya kuhitimu na NSSF kwenye nssf.or.tz.", "correct_answer_en": "Yes. NSSF provides maternity benefits for qualifying contributing female members. Other benefits include retirement pension, disability, survivors, and funeral grant. Confirm eligibility conditions with NSSF at nssf.or.tz.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_102", "subdomain": "nssf_contributions", "question_sw": "Je, deadline ya kulipa michango ya NSSF kila mwezi ni siku ya 20 ya mwezi unaofuata?", "question_en": "Is the monthly NSSF contribution deadline the 20th of the following month?", "correct_answer_sw": "Hapana. Kwa mujibu wa Sheria ya NSSF, michango inalipwa ndani ya mwezi mmoja baada ya kulipa mishahara. Kwa vitendo, NSSF karibu inatarajia malipo ifikapo siku ya 10 ya mwezi unaofuata \\u2014 si siku ya 20. Siku ya 20 ni deadline ya VAT return, tofauti kabisa. Thibitisha tarehe sahihi na nssf.or.tz.", "correct_answer_en": "No. Under the NSSF Act, contributions must be paid within one month of salary payment. In practice, NSSF generally expects payment by the 10th of the following month \\u2014 not the 20th. The 20th is the VAT return deadline, completely different. Confirm the exact date at nssf.or.tz.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_103", "subdomain": "nssf_contributions", "question_sw": "Je, mwajiri anaweza kuahirisha malipo ya NSSF wakati wa msongo wa kifedha bila adhabu?", "question_en": "Can an employer defer NSSF payments during a financial crisis without penalty?", "correct_answer_sw": "Hapana \\u2014 adhabu zinatumika bila kujali sababu. Unachopaswa kufanya ni kuwasiliana na NSSF mapema ili kujadili mpango wa malipo kabla ya muda wa mwisho. Kukimbia bila kuwasiliana kunazidisha tatizo \\u2014 adhabu zinaendelea kukua. Wasiliana na NSSF kwenye nssf.or.tz haraka.", "correct_answer_en": "No \\u2014 penalties apply regardless of the reason for delay. What you should do is contact NSSF proactively before the deadline to discuss a payment arrangement. Going silent worsens the problem \\u2014 penalties accumulate. Contact NSSF at nssf.or.tz as quickly as possible.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_104", "subdomain": "nssf_contributions", "question_sw": "Mwajiri anahitaji nyaraka gani kuwasilisha NSSF ili kusajilisha wafanyakazi wake?", "question_en": "What documents must an employer bring to NSSF to register their employees?", "correct_answer_sw": "Unahitaji: fomu ya usajilishaji wa mwajiri; nakala ya certificate ya BRELA; TIN; vitambulisho vya kitaifa vya wafanyakazi wote. Thibitisha orodha kamili na NSSF kwenye nssf.or.tz kabla ya kwenda.", "correct_answer_en": "You need: employer registration form; copy of BRELA certificate; TIN; national ID documents of all employees. Confirm the complete list with NSSF at nssf.or.tz before going.", "answer_type": "procedure", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_105", "subdomain": "nssf_contributions", "question_sw": "Ninajua sitaweza kulipa NSSF kwa wakati mwezi huu kwa sababu ya tatizo la mtiririko wa pesa \\u2014 nifanye nini?", "question_en": "I cannot pay NSSF on time this month due to cash-flow problems \\u2014 what should I do?", "correct_answer_sw": "Wasiliana na NSSF mara moja \\u2014 kabla ya muda wa mwisho, si baada yake. Eleza hali yako na omba mpango wa malipo. Kukimbia bila kuwasiliana kunazidisha tatizo na adhabu. Thibitisha chaguzi zilizopo na NSSF kwenye nssf.or.tz.", "correct_answer_en": "Contact NSSF immediately \\u2014 before the deadline, not after. Explain your situation and request a payment arrangement. Going silent worsens the problem and penalties. Confirm available options with NSSF at nssf.or.tz.", "answer_type": "procedure", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_106", "subdomain": "nssf_contributions", "question_sw": "NSSF inampa mwajiri nini baada ya usajilishaji kukamilika?", "question_en": "What does NSSF provide to the employer after registration is complete?", "correct_answer_sw": "NSSF inatoa nambari ya akaunti ya mwajiri na maelekezo ya kulipa michango kila mwezi. Kila mfanyakazi aliyesajiliwa anapewa nambari yake ya uanachama wa NSSF. Thibitisha taarifa zinazotolewa na NSSF kwenye nssf.or.tz.", "correct_answer_en": "NSSF issues the employer an account number and monthly contribution payment instructions. Each registered employee is also assigned their own NSSF membership number. Confirm the information provided with NSSF at nssf.or.tz.", "answer_type": "procedure", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_107", "subdomain": "nssf_contributions", "question_sw": "Mfanyabiashara anayejitegemea anachukua hatua gani kujiunga NSSF kama mwanachama wa hiari?", "question_en": "What steps does a self-employed person take to join NSSF as a voluntary member?", "correct_answer_sw": "1) Tembelea ofisi ya NSSF au tovuti nssf.or.tz. 2) Omba fomu ya uanachama wa hiari. 3) Wasilisha fomu pamoja na kitambulisho cha kitaifa. 4) NSSF itakupa nambari ya uanachama. 5) Anza kulipa michango ya kila mwezi. Thibitisha viwango vya mchango wa hiari na NSSF.", "correct_answer_en": "1) Visit an NSSF office or nssf.or.tz. 2) Request the voluntary membership form. 3) Submit the form with your national ID. 4) NSSF will issue your membership number. 5) Begin monthly contributions. Confirm voluntary contribution rates with NSSF.", "answer_type": "procedure", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_108", "subdomain": "nssf_contributions", "question_sw": "NSSF inatoza riba kiasi gani kwenye michango iliyochelewa ya mwajiri?", "question_en": "What interest does NSSF charge on an employer\'s late contributions?", "correct_answer_sw": "NSSF inatoza riba ya asilimia 5 kwa kila mwezi wa ucheleweshaji chini ya Sheria ya NSSF Sura ya 50 kifungu cha 14. Kwa kuongeza riba, NSSF inaweza kuchukua hatua za kisheria. Epuka ucheleweshaji \\u2014 wasiliana na NSSF mapema ukijua hutaweza kulipa.", "correct_answer_en": "NSSF charges 5% interest per month on late contributions under the NSSF Act Cap. 50 s.14. In addition to interest, NSSF can take legal action. Avoid delays \\u2014 contact NSSF early if you know you cannot pay on time.", "answer_type": "penalty", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_109", "subdomain": "nssf_contributions", "question_sw": "Je, mkurugenzi wa kampuni anaweza kuwajibika binafsi kwa michango ya NSSF isiyolipwa?", "question_en": "Can a company director be held personally liable for unpaid NSSF contributions?", "correct_answer_sw": "Ndiyo \\u2014 katika hali mbaya, wakurugenzi wa kampuni wanaweza kukabili wajibu wa kibinafsi. Hii ni hatari kubwa kwa wakurugenzi wa makampuni madogo hasa. Usizuie malipo ya NSSF \\u2014 wasiliana na mshauri wa kisheria au NSSF mara matatizo yanapotokea.", "correct_answer_en": "Yes \\u2014 in serious cases, company directors can face personal liability for unpaid NSSF contributions. This is a significant risk especially for directors of small companies. Do not withhold NSSF payments \\u2014 contact a legal adviser or NSSF immediately when problems arise.", "answer_type": "penalty", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_110", "subdomain": "nssf_contributions", "question_sw": "Mwajiri anayoendelea kutosajilisha wafanyakazi wake NSSF \\u2014 NSSF inaweza kuchukua hatua gani?", "question_en": "An employer persistently failing to register employees with NSSF \\u2014 what action can NSSF take?", "correct_answer_sw": "NSSF inaweza: kufungua kesi ya kisheria; kutoza faini; kutoa amri ya kulazimisha usajilishaji; na katika hali mbaya, kuwashirikisha wakurugenzi kibinafsi. Kutosajilisha ni ukiukwaji mkubwa wa Sheria ya NSSF. Thibitisha na NSSF kwenye nssf.or.tz.", "correct_answer_en": "NSSF can: file a legal case; impose fines; issue an order compelling registration; and in serious cases, hold directors personally liable. Failing to register is a serious NSSF Act violation. Confirm with NSSF at nssf.or.tz.", "answer_type": "penalty", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_111", "subdomain": "sdl_compliance", "question_sw": "Kiwango cha SDL ni asilimia ngapi ya jumla ya mishahara ya jumla ya wafanyakazi wote?", "question_en": "What is the SDL rate as a percentage of total gross payroll?", "correct_answer_sw": "Asilimia 3.5 ya jumla ya mishahara ya jumla (gross wages) ya wafanyakazi wote. SDL inalipwa na mwajiri peke yake \\u2014 haikatwi kwa mfanyakazi. Thibitisha kiwango cha sasa na TRA kwenye tra.go.tz.", "correct_answer_en": "3.5% of total gross wages of all employees. SDL is paid by the employer alone \\u2014 not deducted from employees\' salaries. Confirm the current rate with TRA at tra.go.tz.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_112", "subdomain": "sdl_compliance", "question_sw": "Kiwango cha WCF ni asilimia ngapi ya jumla ya mishahara ya jumla?", "question_en": "What is the WCF rate as a percentage of total gross payroll?", "correct_answer_sw": "Asilimia 0.5 ya jumla ya mishahara ya jumla ya wafanyakazi wote. WCF inalipwa na mwajiri \\u2014 haikatwi mshahara wa mfanyakazi. WCF inalipwa kwa Mamlaka ya WCF, si TRA. Thibitisha kiwango na Mamlaka ya WCF.", "correct_answer_en": "0.5% of total gross wages of all employees. WCF is paid by the employer \\u2014 not deducted from employees\' salaries. WCF is paid to the WCF Authority, not TRA. Confirm the rate with the WCF Authority.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_113", "subdomain": "sdl_compliance", "question_sw": "SDL inaanza kulipwa mwajiri ana wafanyakazi wangapi au zaidi?", "question_en": "SDL becomes payable when an employer has how many or more employees?", "correct_answer_sw": "Wafanyakazi 10 au zaidi. Waajiri wenye wafanyakazi chini ya 10 hawajumuishwi na SDL kwa kawaida. Mara idadi ikifika 10, SDL inatakiwa kulipwa mara moja. Thibitisha na TRA kwenye tra.go.tz.", "correct_answer_en": "10 or more employees. Employers with fewer than 10 employees are generally not subject to SDL. Once the count reaches 10, SDL is payable immediately. Confirm with TRA at tra.go.tz.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_114", "subdomain": "sdl_compliance", "question_sw": "SDL inalipwa pamoja na kodi gani nyingine na inawasilishwa TRA ifikapo tarehe ngapi kila mwezi?", "question_en": "SDL is paid together with which other tax, and by what date must it reach TRA each month?", "correct_answer_sw": "SDL inalipwa pamoja na PAYE ifikapo siku ya 7 ya mwezi unaofuata. Kwa mfano, SDL ya Machi inalipwa ifikapo 7 Aprili. Thibitisha tarehe hii na TRA kwenye tra.go.tz.", "correct_answer_en": "SDL is paid together with PAYE by the 7th of the following month. For example, March SDL is paid by 7 April. Confirm this date with TRA at tra.go.tz.", "answer_type": "number", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_115", "subdomain": "sdl_compliance", "question_sw": "GN 605A ya mishahara ya chini ya sekta binafsi inaanza kutumika tarehe ngapi?", "question_en": "From what date does GN 605A minimum wage order take effect?", "correct_answer_sw": "Tarehe 1 Januari 2026. GN 605A ilitangazwa tarehe 13 Oktoba 2025 na ilianza kutumika 1 Januari 2026. Amri ya mishahara ya 2022 ilifutwa kuanzia tarehe hiyo hiyo. Thibitisha na PKF Eastern Africa au VELMA Law.", "correct_answer_en": "1 January 2026. GN 605A was gazetted 13 October 2025 and took effect 1 January 2026. The 2022 wage order was revoked from the same date. Confirm with PKF Eastern Africa or VELMA Law.", "answer_type": "number", "source_url": "https://www.pkfea.com/publications/2025/new-minimum-wage-order-2025-key-highlights-for-tanzania-s-private-sector/"}, {"id": "eval_116", "subdomain": "sdl_compliance", "question_sw": "Ongezeko la wastani la mishahara ya chini ya sekta binafsi chini ya GN 605A ni asilimia ngapi?", "question_en": "What is the average percentage increase in private sector minimum wages under GN 605A?", "correct_answer_sw": "Asilimia 33.4. Mshahara wa wastani ulipanda kutoka TZS 275,060 hadi TZS 358,322 kwa mwezi. Thibitisha takwimu hizi na PKF Eastern Africa.", "correct_answer_en": "33.4%. The average wage rose from TZS 275,060 to TZS 358,322 per month. Confirm these figures with PKF Eastern Africa.", "answer_type": "number", "source_url": "https://www.pkfea.com/publications/2025/new-minimum-wage-order-2025-key-highlights-for-tanzania-s-private-sector/"}, {"id": "eval_117", "subdomain": "sdl_compliance", "question_sw": "GN 605A inashughulikia sekta ngapi na sekta ndogo ngapi za mishahara ya chini?", "question_en": "How many sectors and sub-sectors does GN 605A cover?", "correct_answer_sw": "Sekta 16 na sekta ndogo (sub-sectors) 46. Kila sub-sector ina kiwango chake maalum cha mshahara wa chini. Thibitisha orodha kamili na PKF Eastern Africa au TanzLII.", "correct_answer_en": "16 sectors and 46 sub-sectors. Each sub-sector has its own specific minimum wage rate. Confirm the complete list with PKF Eastern Africa or TanzLII.", "answer_type": "number", "source_url": "https://www.pkfea.com/publications/2025/new-minimum-wage-order-2025-key-highlights-for-tanzania-s-private-sector/"}, {"id": "eval_118", "subdomain": "sdl_compliance", "question_sw": "Wastani wa mshahara wa chini wa sekta binafsi Tanzania KABLA ya GN 605A kuanza kutumika ulikuwa TZS ngapi kwa mwezi?", "question_en": "What was the average private sector minimum wage BEFORE GN 605A took effect?", "correct_answer_sw": "TZS 275,060 kwa mwezi \\u2014 chini ya amri ya mishahara ya 2022 ambayo ilifutwa kuanzia 1 Januari 2026. Kiwango hiki si halali tena baada ya tarehe hiyo. Thibitisha na PKF Eastern Africa.", "correct_answer_en": "TZS 275,060 per month \\u2014 under the 2022 wage order which was revoked from 1 January 2026. This rate is no longer valid after that date. Confirm with PKF Eastern Africa.", "answer_type": "number", "source_url": "https://www.pkfea.com/publications/2025/new-minimum-wage-order-2025-key-highlights-for-tanzania-s-private-sector/"}, {"id": "eval_119", "subdomain": "sdl_compliance", "question_sw": "Wastani wa mshahara wa chini wa sekta binafsi Tanzania BAADA ya GN 605A kuanza ni TZS ngapi kwa mwezi?", "question_en": "What is the average private sector minimum wage AFTER GN 605A took effect?", "correct_answer_sw": "TZS 358,322 kwa mwezi kwa wastani. Hii ni ongezeko la asilimia 33.4. Kiwango halisi kinatofautiana kwa sekta na sub-sector \\u2014 kina anuwai kutoka karibu TZS 175,000 hadi TZS 765,900 (madini/nishati ya kimataifa). Thibitisha na VELMA Law au PKF.", "correct_answer_en": "TZS 358,322 per month on average. This is a 33.4% increase. The exact rate varies by sector and sub-sector \\u2014 ranging from approximately TZS 175,000 to TZS 765,900 (international mining/energy). Confirm with VELMA Law or PKF.", "answer_type": "number", "source_url": "https://velmalaw.co.tz/news/new-private-sector-minimum-wage-order-from-1-january-2026/"}, {"id": "eval_120", "subdomain": "sdl_compliance", "question_sw": "Kiwango cha juu zaidi cha mshahara wa chini chini ya GN 605A ni TZS ngapi na ni kwa sekta gani?", "question_en": "What is the highest minimum wage rate under GN 605A and for which sector?", "correct_answer_sw": "Karibu TZS 765,900 kwa mwezi \\u2014 kwa sekta ya madini na nishati ya kimataifa (international mining/energy). Kiwango cha chini kabisa ni karibu TZS 175,000. Thibitisha takwimu hizi na PKF Eastern Africa au Clyde & Co.", "correct_answer_en": "Approximately TZS 765,900 per month \\u2014 for the international mining and energy sector. The lowest rate is approximately TZS 175,000. Confirm with PKF Eastern Africa or Clyde & Co.", "answer_type": "number", "source_url": "https://www.clydeco.com/en/insights/2026/02/introduction-of-a-new-wage-order-in-tanzania"}, {"id": "eval_121", "subdomain": "sdl_compliance", "question_sw": "Je, mwajiri mwenye wafanyakazi 8 ana wajibu wa kulipa SDL?", "question_en": "Does an employer with 8 employees have an obligation to pay SDL?", "correct_answer_sw": "Hapana. Kizingiti cha SDL ni wafanyakazi 10 au zaidi. Mwajiri mwenye wafanyakazi 8 bado hajafika kizingiti hicho. Hata hivyo, mara idadi ikifika 10, SDL inatakiwa kulipwa mara moja. Fuatilia idadi kila mwezi. Thibitisha na TRA.", "correct_answer_en": "No. The SDL threshold is 10 or more employees. An employer with 8 employees has not reached that threshold yet. However, once the count reaches 10, SDL is payable immediately. Monitor your count monthly. Confirm with TRA.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_122", "subdomain": "sdl_compliance", "question_sw": "Je, SDL inakatwa kutoka mshahara wa mfanyakazi na kulipwa TRA?", "question_en": "Is SDL deducted from an employee\'s salary and paid to TRA?", "correct_answer_sw": "Hapana. SDL inalipwa na mwajiri peke yake kutoka pesa zake mwenyewe \\u2014 si kukatwa mshahara wa mfanyakazi. Hii ni tofauti na NSSF ambapo sehemu ya mfanyakazi inakatwa mshahara. SDL ni mzigo wa mwajiri kabisa. Thibitisha na TRA.", "correct_answer_en": "No. SDL is paid entirely by the employer from their own funds \\u2014 not deducted from employees\' salaries. This differs from NSSF where the employee\'s share is deducted from salary. SDL is entirely the employer\'s burden. Confirm with TRA.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_123", "subdomain": "sdl_compliance", "question_sw": "Je, WCF na SDL ni ushuru ule ule unaolipwa mahali pamoja?", "question_en": "Are WCF and SDL the same levy paid to the same institution?", "correct_answer_sw": "Hapana. Ni ushuru tofauti kabisa: SDL (asilimia 3.5) inalipwa TRA, inafadhili mafunzo ya ujuzi. WCF (asilimia 0.5) inalipwa Mamlaka ya WCF, inafadhili fidia ya majeruhi wa kazi. Zote mbili zinalipwa na mwajiri lakini kwa taasisi tofauti kwa madhumuni tofauti.", "correct_answer_en": "No. They are completely different levies: SDL (3.5%) paid to TRA, funds skills training. WCF (0.5%) paid to the WCF Authority, funds workplace injury compensation. Both paid by the employer but to different institutions for different purposes.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_124", "subdomain": "sdl_compliance", "question_sw": "Biashara yangu ina wafanyakazi 9 na ninaajiri mfanyakazi wa 10 katikati ya mwezi \\u2014 je, SDL inatakiwa kulipwa mwezi huo huo?", "question_en": "My business has 9 employees and I hire a 10th employee mid-month \\u2014 is SDL payable that same month?", "correct_answer_sw": "Ndiyo. Mara idadi ya wafanyakazi wako ikiwa 10 au zaidi, SDL inatakiwa kulipwa mara moja \\u2014 haijalishi tarehe ya kuajiriwa kwa mfanyakazi wa 10. Thibitisha utaratibu wa mwanzo wa SDL na TRA.", "correct_answer_en": "Yes. Once your employee count reaches 10 or more, SDL is payable immediately \\u2014 regardless of what date the 10th employee was hired. Confirm the SDL commencement procedure with TRA.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_125", "subdomain": "sdl_compliance", "question_sw": "Je, amri ya mishahara ya chini ya sekta binafsi ya mwaka 2022 bado ina nguvu tangu Januari 2026?", "question_en": "Is the 2022 private sector minimum wage order still valid since January 2026?", "correct_answer_sw": "Hapana. Amri ya mishahara ya 2022 ilifutwa rasmi kuanzia 1 Januari 2026. GN 605A ndiyo amri mpya inayotumika. Mwajiri yeyote anayetumia viwango vya 2022 baada ya tarehe hiyo anachukuliwa kuwa hana utiifu wa kisheria. Thibitisha na PKF Eastern Africa au VELMA Law.", "correct_answer_en": "No. The 2022 minimum wage order was formally revoked from 1 January 2026. GN 605A is the current binding order. Any employer using 2022 rates after that date is non-compliant with the law. Confirm with PKF Eastern Africa or VELMA Law.", "answer_type": "yes_no", "source_url": "https://www.pkfea.com/publications/2025/new-minimum-wage-order-2025-key-highlights-for-tanzania-s-private-sector/"}, {"id": "eval_126", "subdomain": "sdl_compliance", "question_sw": "Je, mshahara wa chini wa TZS 500,000 kwa sekta ya umma umetokana na GN 605A?", "question_en": "Does the TZS 500,000 public sector minimum wage come from GN 605A?", "correct_answer_sw": "Hapana. Mshahara wa TZS 500,000 kwa sekta ya umma ulitangazwa na Rais Samia tarehe 1 Mei 2025, ukianza Julai 2025 \\u2014 ni agizo tofauti la serikali, si GN 605A. GN 605A inashughulikia sekta binafsi peke yake. Thibitisha tofauti hizi na Clyde & Co au VELMA Law. Thibitisha tofauti hii na VELMA Law au Clyde and Co ambao wameandika uchambuzi wa kina wa mabadiliko haya.", "correct_answer_en": "No. The TZS 500,000 for the public sector was announced by President Samia on 1 May 2025, effective July 2025 \\u2014 a separate government directive, not GN 605A. GN 605A covers the private sector only. Confirm these distinctions with Clyde & Co or VELMA Law. Confirm this distinction with VELMA Law or Clyde and Co who have published detailed analysis of these changes.", "answer_type": "yes_no", "source_url": "https://www.clydeco.com/en/insights/2026/02/introduction-of-a-new-wage-order-in-tanzania"}, {"id": "eval_127", "subdomain": "sdl_compliance", "question_sw": "Je, SDL na PAYE zinalipwa TRA kwa wakati mmoja \\u2014 siku ya 7 ya mwezi unaofuata?", "question_en": "Are SDL and PAYE both due to TRA on the same date \\u2014 the 7th of the following month?", "correct_answer_sw": "Ndiyo. SDL na PAYE zote mbili zinalipwa TRA ifikapo siku ya 7 ya mwezi unaofuata. Thibitisha tarehe hii na TRA kwenye tra.go.tz kwani inaweza kubadilika.", "correct_answer_en": "Yes. Both SDL and PAYE are due to TRA by the 7th of the following month. Confirm this date with TRA at tra.go.tz as it may change.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_128", "subdomain": "sdl_compliance", "question_sw": "Je, mwajiri anaweza kulipa WCF moja kwa moja TRA badala ya Mamlaka ya WCF?", "question_en": "Can an employer pay WCF directly to TRA instead of the WCF Authority?", "correct_answer_sw": "Hapana. WCF inalipwa kwa Mamlaka ya WCF, si TRA. Kulipa TRA badala ya Mamlaka ya WCF hakutakuridhisha wajibu wako wa WCF. Thibitisha njia sahihi ya malipo ya WCF na Mamlaka ya WCF.", "correct_answer_en": "No. WCF is paid to the WCF Authority, not TRA. Paying TRA instead of the WCF Authority will not fulfil your WCF obligation. Confirm the correct WCF payment method with the WCF Authority.", "answer_type": "yes_no", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_129", "subdomain": "sdl_compliance", "question_sw": "Mpangilio sahihi wa makato ya mishahara na malipo ya kodi za mwajiri kwa mwezi ni upi?", "question_en": "What is the correct sequence for payroll deductions and employer tax payments each month?", "correct_answer_sw": "1) Kata NSSF ya mfanyakazi (10%) na PAYE kutoka mshahara wa jumla. 2) Lipa mshahara wa mkono kwa mfanyakazi. 3) Peleka PAYE + SDL TRA ifikapo siku ya 7 ya mwezi unaofuata. 4) Peleka NSSF jumla (20%) kwa NSSF ndani ya mwezi mmoja baada ya kulipa mishahara (kwa vitendo ifikapo siku ya 10). 5) Peleka WCF kwa Mamlaka ya WCF. Thibitisha mpangilio huu na washauri wa kodi.", "correct_answer_en": "1) Deduct employee NSSF (10%) and PAYE from gross salary. 2) Pay net salary to the employee. 3) Remit PAYE + SDL to TRA by the 7th of the following month. 4) Remit NSSF total (20%) to NSSF by the 10th. 5) Remit WCF to the WCF Authority. Confirm this sequence with tax advisers.", "answer_type": "procedure", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_130", "subdomain": "sdl_compliance", "question_sw": "Mwajiri anahesabu kiasi cha SDL cha kulipa kwa mwezi vipi?", "question_en": "How does an employer calculate the SDL amount payable each month?", "correct_answer_sw": "Zidisha jumla ya mishahara ya jumla (gross wages) ya wafanyakazi wote kwa asilimia 3.5. Mfano: jumla ya mishahara ya wafanyakazi wote ni TZS milioni 10 \\u2014 SDL = 3.5% \\u00d7 10,000,000 = TZS 350,000. Thibitisha hesabu yako na mshauri wa kodi au TRA.", "correct_answer_en": "Multiply total gross wages of all employees by 3.5%. Example: total payroll is TZS 10 million \\u2014 SDL = 3.5% \\u00d7 10,000,000 = TZS 350,000. Confirm your calculation with a tax adviser or TRA.", "answer_type": "procedure", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_131", "subdomain": "sdl_compliance", "question_sw": "Mwajiri anagundua vipi kiwango cha mshahara wa chini cha GN 605A kinachofaa kwa kila mfanyakazi wake?", "question_en": "How does an employer determine which GN 605A minimum wage rate applies to each of their workers?", "correct_answer_sw": "Bainisha sekta na sub-sector inayohusu biashara yako kutoka orodha ya GN 605A (sekta 16, sub-sectors 46). Kila sub-sector ina kiwango chake maalum. Soma GN 605A kwenye TanzLII au omba ushauri kutoka PKF Eastern Africa au VELMA Law kwa sekta yako maalum.", "correct_answer_en": "Identify the sector and sub-sector relevant to your business from the GN 605A list (16 sectors, 46 sub-sectors). Each sub-sector has its own specific rate. Read GN 605A at TanzLII or seek advice from PKF Eastern Africa or VELMA Law for your specific sector.", "answer_type": "procedure", "source_url": "https://velmalaw.co.tz/news/new-private-sector-minimum-wage-order-from-1-january-2026/"}, {"id": "eval_132", "subdomain": "sdl_compliance", "question_sw": "Nimegundua nimekuwa nikalipa mishahara chini ya viwango vya GN 605A tangu Januari 2026 \\u2014 nifanye nini?", "question_en": "I discover I have been paying wages below GN 605A rates since January 2026 \\u2014 what should I do?", "correct_answer_sw": "1) Hesabu backpay kwa kila mfanyakazi kuanzia 1 Januari 2026. 2) Lipa tofauti hiyo haraka iwezekanavyo. 3) Sasisisha mishahara yote kulingana na viwango vya GN 605A mara moja. 4) Wasiliana na mshauri wa kisheria kama kuna hatari ya malalamiko ya wafanyakazi. Kutolipa mishahara inayostahili ni ukiukwaji wa kisheria.", "correct_answer_en": "1) Calculate backpay for each employee from 1 January 2026. 2) Pay the difference as quickly as possible. 3) Update all wages to GN 605A rates immediately. 4) Consult a legal adviser if there is risk of employee complaints. Failing to pay entitled wages is a legal violation.", "answer_type": "procedure", "source_url": "https://www.pkfea.com/publications/2025/new-minimum-wage-order-2025-key-highlights-for-tanzania-s-private-sector/"}, {"id": "eval_133", "subdomain": "sdl_compliance", "question_sw": "Matokeo ya kisheria ya mwajiri anayolipa mshahara chini ya viwango vya GN 605A ni yapi?", "question_en": "What are the legal consequences for an employer paying wages below GN 605A rates?", "correct_answer_sw": "Ukiukwaji wa sheria ya kazi Tanzania. Mfanyakazi anaweza kuleta malalamiko kwa CMA (Commission for Mediation and Arbitration). Mwajiri anaweza kulazimishwa kulipa mishahara iliyobaki pamoja na fidia. Katika hali mbaya, kesi ya Mahakama ya Kazi inaweza kufuata. Hakikisha mishahara inakidhi viwango vya GN 605A kwa sekta yako.", "correct_answer_en": "A violation of Tanzania labour law. The employee can file a complaint with the CMA (Commission for Mediation and Arbitration). The employer may be ordered to pay outstanding wages plus compensation. In serious cases, a Labour Court case can follow. Ensure wages meet GN 605A rates for your sector.", "answer_type": "penalty", "source_url": "https://velmalaw.co.tz/news/new-private-sector-minimum-wage-order-from-1-january-2026/"}, {"id": "eval_134", "subdomain": "sdl_compliance", "question_sw": "TRA inatoza adhabu gani kwa mwajiri aliyechelewa kulipa SDL?", "question_en": "What penalties does TRA impose for late SDL payment?", "correct_answer_sw": "TRA inatoza faini na riba kwenye SDL iliyochelewa. Kiasi maalum kinapaswa kuthibitishwa moja kwa moja na TRA kwenye tra.go.tz kwani kinaweza kubadilika baada ya Finance Act ya kila mwaka. Lipa SDL ikiwa na PAYE ifikapo siku ya 7 ili kuepuka adhabu.", "correct_answer_en": "TRA imposes fines and interest on late SDL. The specific amounts must be confirmed directly with TRA at tra.go.tz as they may change after each annual Finance Act. Pay SDL together with PAYE by the 7th to avoid penalties.", "answer_type": "penalty", "source_url": "https://taxsummaries.pwc.com/tanzania/corporate/other-taxes"}, {"id": "eval_135", "subdomain": "sdl_compliance", "question_sw": "Mwajiri anadai viwango vya amri ya mishahara ya 2022 bado vinatumika kwa wafanyakazi wake mwaka 2026 \\u2014 hali yake ya kisheria ni nini?", "question_en": "An employer claims 2022 wage order rates still apply in 2026 \\u2014 what is their legal position?", "correct_answer_sw": "Hana utiifu wa kisheria. Amri ya mishahara ya 2022 ilifutwa rasmi kuanzia 1 Januari 2026 na GN 605A. Mwajiri anayetumia viwango vya 2022 baada ya tarehe hiyo anakiuka sheria ya kazi na anaweza kukabiliwa na malalamiko ya wafanyakazi na hatua za kisheria. Thibitisha na VELMA Law au PKF.", "correct_answer_en": "They are non-compliant. The 2022 wage order was formally revoked from 1 January 2026 by GN 605A. An employer using 2022 rates after that date violates labour law and can face employee complaints and legal action. Confirm with VELMA Law or PKF.", "answer_type": "penalty", "source_url": "https://www.pkfea.com/publications/2025/new-minimum-wage-order-2025-key-highlights-for-tanzania-s-private-sector/"}, {"id": "eval_136", "subdomain": "gn487a", "question_sw": "GN 487A inazuia aina ngapi za biashara kwa wageni?", "question_en": "How many categories of business does GN 487A prohibit for non-citizens?", "correct_answer_sw": "GN 487A inakataza aina 15 za biashara kwa wageni.", "correct_answer_en": "GN 487A prohibits 15 categories of business for non-citizens.", "answer_type": "number", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_137", "subdomain": "gn487a", "question_sw": "Faini ya chini kabisa kwa mgeni anayefanya biashara iliyokatazwa ni TZS ngapi?", "question_en": "What is the minimum fine for a non-citizen caught conducting a prohibited business?", "correct_answer_sw": "Faini ya chini kabisa ni TZS milioni 10.", "correct_answer_en": "The minimum fine is TZS 10 million.", "answer_type": "number", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_138", "subdomain": "gn487a", "question_sw": "Mgeni aliyekamatwa akifanya biashara iliyokatazwa anaweza kufungwa miezi mingapi?", "question_en": "How many months of imprisonment can a non-citizen face for conducting a prohibited business?", "correct_answer_sw": "Mgeni anaweza kufungwa hadi miezi 6.", "correct_answer_en": "A non-citizen can face up to 6 months of imprisonment.", "answer_type": "number", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_139", "subdomain": "gn487a", "question_sw": "Mtanzania anayenusuru mgeni kufanya biashara haramu atalipia faini ya TZS ngapi?", "question_en": "What is the fine for a Tanzanian who facilitates a non-citizen in conducting a prohibited business?", "correct_answer_sw": "Mtanzania anayenusuru mgeni atalipa faini ya TZS milioni 5.", "correct_answer_en": "A Tanzanian who facilitates a non-citizen faces a fine of TZS 5 million.", "answer_type": "number", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_140", "subdomain": "gn487a", "question_sw": "Mtanzania anayenusuru mgeni kufanya biashara haramu anaweza kufungwa miezi mingapi?", "question_en": "How many months of imprisonment can a Tanzanian facilitator face?", "correct_answer_sw": "Mtanzania anayenusuru mgeni anaweza kufungwa hadi miezi 3.", "correct_answer_en": "A Tanzanian facilitator can face up to 3 months of imprisonment.", "answer_type": "number", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_141", "subdomain": "gn487a", "question_sw": "GN 487A ilianza kutumika tarehe ngapi?", "question_en": "On what date did GN 487A come into effect?", "correct_answer_sw": "GN 487A ilianza kutumika tarehe 28 Julai 2025.", "correct_answer_en": "GN 487A came into effect on 28 July 2025.", "answer_type": "number", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_142", "subdomain": "gn487a", "question_sw": "Zoezi la utekelezaji wa GN 487A lilianza tarehe ngapi?", "question_en": "On what date did the GN 487A enforcement exercise begin?", "correct_answer_sw": "Zoezi la utekelezaji lilianza tarehe 11 Septemba 2025.", "correct_answer_en": "The enforcement exercise began on 11 September 2025.", "answer_type": "number", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_143", "subdomain": "gn487a", "question_sw": "Zoezi la utekelezaji wa GN 487A liliishia tarehe ngapi?", "question_en": "On what date did the GN 487A enforcement exercise end?", "correct_answer_sw": "Zoezi la utekelezaji liliishia tarehe 8 Oktoba 2025.", "correct_answer_en": "The enforcement exercise ended on 8 October 2025.", "answer_type": "number", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_144", "subdomain": "gn487a", "question_sw": "Mke wangu ni Mtanzania \\u2014 kama mgeni mwenye mwenzi wa Kitanzania, GN 487A haitanigusia?", "question_en": "My spouse is Tanzanian \\u2014 as a non-citizen married to a Tanzanian, GN 487A does not apply to me?", "correct_answer_sw": "La. GN 487A bado inakugusia. Mgeni mwenye mwenzi wa Kitanzania bado anachukuliwa mgeni chini ya amri hii.", "correct_answer_en": "No. GN 487A still applies to you. A non-citizen with a Tanzanian spouse is still considered a non-citizen under this order.", "answer_type": "yes_no", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_145", "subdomain": "gn487a", "question_sw": "Naweza kumwomba rafiki yangu Mtanzania asajili biashara kwa jina lake, niendelee mimi kuendeshea \\u2014 inafanya kazi?", "question_en": "Can I ask my Tanzanian friend to register the business in their name while I continue running it?", "correct_answer_sw": "Hapana. Mgeni hawezi kuendesha biashara iliyokatazwa kwa kutumia jina la Mtanzania. Mtanzania huyo atapata adhabu kama msaidizi: faini ya TZS milioni 5 au kifungo cha miezi 3.", "correct_answer_en": "No. A non-citizen cannot continue a prohibited business through a Tanzanian nominee. That Tanzanian will face the facilitator penalty: TZS 5 million or 3 months imprisonment.", "answer_type": "yes_no", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_146", "subdomain": "gn487a", "question_sw": "Serikali ilitoa muda wa kujiandaa baada ya GN 487A kutangazwa mnamo Julai 2025?", "question_en": "Did the government provide a grace period after GN 487A was gazetted in July 2025?", "correct_answer_sw": "Hapana. GN 487A haikutoa muda wa kujiandaa \\u2014 ilianza kutumika mara moja tarehe 28 Julai 2025.", "correct_answer_en": "No. GN 487A did not include a grace period \\u2014 it took effect immediately on 28 July 2025.", "answer_type": "yes_no", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_147", "subdomain": "gn487a", "question_sw": "Biashara ya rejareja \\u2014 kama duka la nguo \\u2014 iko kwenye orodha ya biashara zilizokatazwa kwa wageni?", "question_en": "Is retail trade \\u2014 like a clothing shop \\u2014 on the list of prohibited businesses for non-citizens?", "correct_answer_sw": "Ndiyo. Biashara ya rejareja ipo kwenye orodha ya biashara 15 zilizokatazwa kwa wageni chini ya GN 487A.", "correct_answer_en": "Yes. Retail trade is included in the 15 categories of business prohibited for non-citizens under GN 487A.", "answer_type": "yes_no", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_148", "subdomain": "gn487a", "question_sw": "Biashara ya jumla \\u2014 kuuza bidhaa kwa maduka mengine \\u2014 imekatazwa kwa wageni?", "question_en": "Is wholesale trade \\u2014 selling goods to other shops \\u2014 prohibited for non-citizens?", "correct_answer_sw": "Ndiyo. Biashara ya jumla imekatazwa kwa wageni chini ya GN 487A.", "correct_answer_en": "Yes. Wholesale trade is prohibited for non-citizens under GN 487A.", "answer_type": "yes_no", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_149", "subdomain": "gn487a", "question_sw": "Rafiki yangu wa Ethiopia ana duka la kutengeneza simu Kariakoo \\u2014 GN 487A inamfunika?", "question_en": "My Ethiopian friend has a phone repair shop in Kariakoo \\u2014 does GN 487A cover him?", "correct_answer_sw": "Ndiyo. Utengenezaji wa simu uko kwenye orodha ya biashara zilizokatazwa, hivyo rafiki yako anafunikwa na GN 487A.", "correct_answer_en": "Yes. Phone repair is on the list of prohibited activities, so your friend is covered by GN 487A.", "answer_type": "yes_no", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_150", "subdomain": "gn487a", "question_sw": "Saluni yangu ya nywele \\u2014 mimi ni mgeni \\u2014 imekatazwa na GN 487A?", "question_en": "My hair salon \\u2014 I am a non-citizen \\u2014 is it prohibited under GN 487A?", "correct_answer_sw": "Ndiyo. Biashara ya saluni na urembo imekatazwa kwa wageni chini ya GN 487A.", "correct_answer_en": "Yes. Salon and beauty business is prohibited for non-citizens under GN 487A.", "answer_type": "yes_no", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_151", "subdomain": "gn487a", "question_sw": "Kama wakala wa mobile money \\u2014 mimi ni mgeni \\u2014 GN 487A inanikataza?", "question_en": "As a mobile money agent \\u2014 I am a non-citizen \\u2014 does GN 487A prohibit me?", "correct_answer_sw": "Ndiyo. Uhamishaji wa pesa za simu uko kwenye orodha ya biashara zilizokatazwa kwa wageni chini ya GN 487A.", "correct_answer_en": "Yes. Mobile money transfers are on the list of prohibited activities for non-citizens under GN 487A.", "answer_type": "yes_no", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_152", "subdomain": "gn487a", "question_sw": "Kampuni yangu ya usafi wa nyumba na ofisi \\u2014 mimi si raia wa Tanzania \\u2014 inakabiliwa na GN 487A?", "question_en": "My home and office cleaning company \\u2014 I am not a Tanzanian citizen \\u2014 is it affected by GN 487A?", "correct_answer_sw": "Ndiyo. Huduma za usafi wa nyumba na ofisi zimekatazwa kwa wageni chini ya GN 487A.", "correct_answer_en": "Yes. Home and office cleaning services are prohibited for non-citizens under GN 487A.", "answer_type": "yes_no", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_153", "subdomain": "gn487a", "question_sw": "Mimi ni raia wa Tanzania \\u2014 ninaweza kuendelea kufanya biashara ya rejareja bila tatizo la GN 487A?", "question_en": "I am a Tanzanian citizen \\u2014 can I continue doing retail business without any problem under GN 487A?", "correct_answer_sw": "Ndiyo. GN 487A inawahusu wageni tu. Raia wa Tanzania wanaweza kufanya biashara zote zilizoorodheshwa bila tatizo.", "correct_answer_en": "Yes. GN 487A applies to non-citizens only. Tanzanian citizens can conduct all the listed businesses without issue.", "answer_type": "yes_no", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_154", "subdomain": "gn487a", "question_sw": "Kama mgeni amekamatwa akifanya biashara iliyokatazwa, kibali chake cha kuingia Tanzania kinafutwa?", "question_en": "If a non-citizen is caught conducting a prohibited business, is their visa/entry permit revoked?", "correct_answer_sw": "Ndiyo. Adhabu kwa mgeni inajumuisha kufutwa kwa kibali chake cha kuingia Tanzania.", "correct_answer_en": "Yes. The penalty for a non-citizen includes revocation of their visa/entry permit.", "answer_type": "yes_no", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_155", "subdomain": "gn487a", "question_sw": "Zoezi la utekelezaji wa GN 487A bado linaendelea \\u2014 lazima niwe makini sana sasa hivi?", "question_en": "Is the GN 487A enforcement exercise still ongoing \\u2014 should I be extra careful right now?", "correct_answer_sw": "Hapana. Zoezi la utekelezaji liliishia tarehe 8 Oktoba 2025. Lakini sheria bado ipo na inatekelezwa kudumu \\u2014 si zoezi la muda tu.", "correct_answer_en": "No. The enforcement exercise ended on 8 October 2025. However, the law is permanent and continues to be enforced \\u2014 it was not a temporary campaign only.", "answer_type": "yes_no", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_156", "subdomain": "gn487a", "question_sw": "Jina kamili la GN 487A ni nini?", "question_en": "What is the full name of GN 487A?", "correct_answer_sw": "Jina kamili ni \'Business Licensing (Prohibition of Business Activities for Non-Citizens) Order, 2025\'.", "correct_answer_en": "The full name is \'Business Licensing (Prohibition of Business Activities for Non-Citizens) Order, 2025\'.", "answer_type": "definition", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_157", "subdomain": "gn487a", "question_sw": "GN 487A inasema nini hasa \\u2014 inazuia nini kwa wageni?", "question_en": "What exactly does GN 487A say \\u2014 what does it prohibit for non-citizens?", "correct_answer_sw": "GN 487A inakataza wageni kufanya aina 15 za biashara Tanzania, ikiwemo biashara ya rejareja, jumla, uhamishaji wa pesa za simu, utengenezaji wa simu, saluni, na usafi wa nyumba na ofisi.", "correct_answer_en": "GN 487A prohibits non-citizens from conducting 15 categories of business in Tanzania, including retail trade, wholesale trade, mobile money transfers, phone repair, salon business, and home/office cleaning.", "answer_type": "definition", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_158", "subdomain": "gn487a", "question_sw": "Katika GN 487A, \'msaidizi\' (facilitator) ni nani?", "question_en": "In the context of GN 487A, who is a \'facilitator\'?", "correct_answer_sw": "Msaidizi ni Mtanzania anayenusuru mgeni kufanya biashara iliyokatazwa, kwa mfano kwa kukopa jina lake au leseni yake kwa mgeni.", "correct_answer_en": "A facilitator is a Tanzanian who assists a non-citizen in conducting a prohibited business, for example by lending their name or business licence to the non-citizen.", "answer_type": "definition", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_159", "subdomain": "gn487a", "question_sw": "GN 487A ilipigwa marufuku tarehe gani na ilitekelezwa lini \\u2014 ni tarehe moja au mbili tofauti?", "question_en": "When was GN 487A gazetted and when did it take effect \\u2014 are these the same date or different?", "correct_answer_sw": "Ni tarehe moja. GN 487A ilipigwa marufuku na kuanza kutumika siku ile ile \\u2014 tarehe 28 Julai 2025.", "correct_answer_en": "It is the same date. GN 487A was both gazetted and took effect on the same day \\u2014 28 July 2025.", "answer_type": "definition", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_160", "subdomain": "gn487a", "question_sw": "Chombo gani cha serikali kiliongoza zoezi la utekelezaji wa GN 487A Septemba-Oktoba 2025?", "question_en": "Which government body led the GN 487A enforcement exercise in September-October 2025?", "correct_answer_sw": "Zoezi la utekelezaji liliongozwa na Idara ya Huduma za Uhamiaji (Immigration Services Department).", "correct_answer_en": "The enforcement exercise was led by the Immigration Services Department.", "answer_type": "definition", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_161", "subdomain": "gn487a", "question_sw": "GN 487A inataja \'biashara ya jumla\' na \'biashara ya rejareja\' \\u2014 tofauti yao ni nini?", "question_en": "GN 487A mentions \'wholesale trade\' and \'retail trade\' \\u2014 what is the difference?", "correct_answer_sw": "Biashara ya jumla ni kuuza bidhaa kwa wingi kwa wafanyabiashara wengine. Biashara ya rejareja ni kuuza bidhaa moja kwa moja kwa wateja. Zote mbili zimekatazwa kwa wageni chini ya GN 487A.", "correct_answer_en": "Wholesale trade is selling goods in bulk to other businesses. Retail trade is selling goods directly to end consumers. Both are prohibited for non-citizens under GN 487A.", "answer_type": "definition", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_162", "subdomain": "gn487a", "question_sw": "\'Mgeni\' katika GN 487A inamaanisha nani hasa?", "question_en": "What does \'non-citizen\' mean specifically in GN 487A?", "correct_answer_sw": "Mgeni ni mtu yeyote ambaye si raia wa Tanzania. Hii inajumuisha watu wenye mwenzi wa Kitanzania \\u2014 ndoa na raia wa Tanzania haibadilishi hadhi ya uraia chini ya amri hii.", "correct_answer_en": "A non-citizen is any person who is not a Tanzanian citizen. This includes people married to Tanzanian citizens \\u2014 being married to a Tanzanian does not change citizenship status under this order.", "answer_type": "definition", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_163", "subdomain": "gn487a", "question_sw": "GN 487A inasema nini kuhusu mpango wa \'jina la mkopo\' (nominee) \\u2014 kwanini unazuiwa?", "question_en": "What does GN 487A say about a \'nominee\' arrangement \\u2014 why is it prohibited?", "correct_answer_sw": "Kutumia jina la Mtanzania kufanya biashara iliyokatazwa kunazuiwa. Mgeni bado anachukuliwa kuendesha biashara hiyo. Mtanzania anayekopa jina lake atalipwa adhabu ya msaidizi: faini ya TZS milioni 5 au kifungo cha miezi 3.", "correct_answer_en": "Using a Tanzanian\'s name to conduct a prohibited business is prohibited. The non-citizen is still considered to be running the business. The Tanzanian who lends their name faces the facilitator penalty: a TZS 5 million fine or 3 months imprisonment.", "answer_type": "definition", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_164", "subdomain": "gn487a", "question_sw": "Adhabu kamili kwa mgeni anayekamatwa akifanya biashara ya rejareja ni nini?", "question_en": "What is the full penalty for a non-citizen caught conducting retail trade?", "correct_answer_sw": "Adhabu kamili ni: faini ya angalau TZS milioni 10, hadi miezi 6 jela, na kufutwa kwa kibali chake cha kuingia Tanzania.", "correct_answer_en": "The full penalty is: a minimum fine of TZS 10 million, up to 6 months imprisonment, and revocation of their entry permit/visa.", "answer_type": "penalty", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_165", "subdomain": "gn487a", "question_sw": "Jirani yangu Mtanzania amekopa leseni yake kwa mfanyabiashara wa China \\u2014 adhabu yake ni nini?", "question_en": "My Tanzanian neighbour lent their business licence to a Chinese trader \\u2014 what is their penalty?", "correct_answer_sw": "Jirani yako atakabiliwa na adhabu ya msaidizi: faini ya TZS milioni 5 au kifungo cha hadi miezi 3.", "correct_answer_en": "Your neighbour faces the facilitator penalty: a fine of TZS 5 million or up to 3 months imprisonment.", "answer_type": "penalty", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_166", "subdomain": "gn487a", "question_sw": "Kama nililipa faini ya TZS milioni 10, naweza kuendelea na biashara yangu ya saluni kama mgeni?", "question_en": "If I paid the TZS 10 million fine, can I continue running my salon as a non-citizen?", "correct_answer_sw": "Hapana. Kulipa faini si ruhusa ya kuendelea. Adhabu pia inajumuisha kufutwa kwa visa yako \\u2014 hivyo hutaruhusiwa kubaki Tanzania kuendesha biashara.", "correct_answer_en": "No. Paying the fine does not allow you to continue. The penalty also includes revocation of your visa \\u2014 you will not be permitted to remain in Tanzania to operate the business.", "answer_type": "penalty", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_167", "subdomain": "gn487a", "question_sw": "Mgeni aliyebanwa akitumia jina la Mtanzania kuendesha duka \\u2014 adhabu inabeba yeye au yule Mtanzania?", "question_en": "A non-citizen caught using a Tanzanian\'s name to run a shop \\u2014 does the penalty fall on the non-citizen or the Tanzanian?", "correct_answer_sw": "Adhabu inabeba wote wawili. Mgeni anapata faini ya angalau TZS milioni 10, hadi miezi 6 jela, na kufutwa kwa visa. Yule Mtanzania anapata adhabu ya msaidizi: TZS milioni 5 au miezi 3 jela.", "correct_answer_en": "The penalty falls on both. The non-citizen faces a minimum TZS 10 million fine, up to 6 months imprisonment, and visa revocation. The Tanzanian faces the facilitator penalty: TZS 5 million or 3 months imprisonment.", "answer_type": "penalty", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_168", "subdomain": "gn487a", "question_sw": "Visa ya mgeni inafutwa kama sehemu ya adhabu ya GN 487A \\u2014 au ni uamuzi tofauti wa mahakama?", "question_en": "Is visa revocation part of the GN 487A penalty, or is it a separate court decision?", "correct_answer_sw": "Kufutwa kwa visa ni sehemu ya adhabu inayotajwa na GN 487A \\u2014 si uamuzi wa ziada wa mahakama.", "correct_answer_en": "Visa revocation is part of the penalty specified by GN 487A \\u2014 it is not a separate additional court decision.", "answer_type": "penalty", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_169", "subdomain": "gn487a", "question_sw": "Mtanzania msaidizi atalipa faini ya TZS milioni 5 NA kufungwa miezi 3, au ni moja tu kati ya hizo?", "question_en": "Does a Tanzanian facilitator pay the TZS 5 million fine AND serve 3 months imprisonment, or just one of the two?", "correct_answer_sw": "Ni moja au nyingine. Adhabu ya msaidizi ni TZS milioni 5 AU kifungo cha hadi miezi 3 \\u2014 si lazima zote mbili.", "correct_answer_en": "It is one or the other. The facilitator penalty is TZS 5 million OR up to 3 months imprisonment \\u2014 not necessarily both.", "answer_type": "penalty", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_170", "subdomain": "gn487a", "question_sw": "Faini ya mgeni (TZS milioni 10 kiwango cha chini) ni kubwa zaidi au ndogo kuliko faini ya msaidizi Mtanzania (TZS milioni 5)?", "question_en": "Is the non-citizen fine (TZS 10 million minimum) larger or smaller than the Tanzanian facilitator fine (TZS 5 million)?", "correct_answer_sw": "Faini ya mgeni ni kubwa zaidi \\u2014 kiwango cha chini ni TZS milioni 10, ikilinganishwa na TZS milioni 5 kwa msaidizi Mtanzania.", "correct_answer_en": "The non-citizen fine is larger \\u2014 the minimum is TZS 10 million, compared to TZS 5 million for the Tanzanian facilitator.", "answer_type": "penalty", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_171", "subdomain": "gn487a", "question_sw": "Mimi ni mgeni ninayefanya biashara ya rejareja \\u2014 GN 487A tayari imetangazwa. Nifanye nini sasa?", "question_en": "I am a non-citizen currently doing retail business \\u2014 GN 487A has already been gazetted. What should I do now?", "correct_answer_sw": "Mgeni anayefanya biashara iliyokatazwa anapaswa kusimamisha shughuli hiyo haraka iwezekanavyo. Kama una leseni iliyotolewa kabla ya GN 487A (kabla ya 28 Julai 2025), unaweza kuendelea hadi leseni hiyo iishe muda wake \\u2014 lakini renewal haitaruhusiwa. Biashara mpya au zilizosajiliwa baada ya tarehe hiyo lazima zisimame mara moja. Tafuta ushauri wa kisheria haraka.", "correct_answer_en": "A non-citizen operating a prohibited business should cease operations as soon as possible. If you hold a valid licence issued before GN 487A (before 28 July 2025), you may continue until that licence expires \\u2014 but renewal will not be permitted. New or post-date businesses must stop immediately. Seek legal advice urgently.", "answer_type": "procedure", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_172", "subdomain": "gn487a", "question_sw": "Chombo gani cha serikali kinaendesha utekelezaji wa GN 487A?", "question_en": "Which government body enforces GN 487A?", "correct_answer_sw": "Idara ya Huduma za Uhamiaji (Immigration Services Department) ndiyo inayoendesha utekelezaji wa GN 487A.", "correct_answer_en": "The Immigration Services Department leads the enforcement of GN 487A.", "answer_type": "procedure", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_173", "subdomain": "gn487a", "question_sw": "Kama mgeni, naweza kuanzisha kampuni Tanzania na kumiliki hisa \\u2014 hii inaruhusiwa chini ya GN 487A?", "question_en": "As a non-citizen, can I set up a company in Tanzania and own shares \\u2014 is this permitted under GN 487A?", "correct_answer_sw": "GN 487A inakataza wageni kufanya biashara fulani moja kwa moja. Kumiliki hisa katika kampuni si njia ya mkato ya kukwepa amri hii \\u2014 kutumia Mtanzania kama msimamizi wa kuchezea bado kunakabiliwa na tatizo la msaidizi. Tafuta ushauri wa wakili wa biashara.", "correct_answer_en": "GN 487A prohibits non-citizens from directly conducting certain businesses. Owning shares in a company is not a shortcut to avoid this order \\u2014 using a Tanzanian as a nominee director still faces the facilitator issue. Seek advice from a business lawyer.", "answer_type": "procedure", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_174", "subdomain": "gn487a", "question_sw": "Nilikuwa sijui kuhusu GN 487A \\u2014 polisi walisimamisha duka langu Septemba 2025. Nini kilifanyika?", "question_en": "I did not know about GN 487A \\u2014 the authorities stopped my shop in September 2025. What happened?", "correct_answer_sw": "Idara ya Uhamiaji iliongoza zoezi la utekelezaji kuanzia tarehe 11 Septemba hadi 8 Oktoba 2025. GN 487A ilianza kutumika tarehe 28 Julai 2025 \\u2014 kukosa ujuzi hakukuwa ulinzi wa kisheria.", "correct_answer_en": "The Immigration Services Department conducted an enforcement exercise from 11 September to 8 October 2025. GN 487A took effect on 28 July 2025 \\u2014 lack of knowledge was not a legal defence.", "answer_type": "procedure", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_175", "subdomain": "gn487a", "question_sw": "Ninaoa Mtanzania \\u2014 ni hatua gani za kisheria ambazo zingenipa ruhusa kufanya biashara ya rejareja?", "question_en": "I am marrying a Tanzanian \\u2014 what legal steps would give me permission to conduct retail business?", "correct_answer_sw": "Ndoa na raia wa Tanzania haibadilishi hadhi ya uraia chini ya GN 487A \\u2014 bado utachukuliwa mgeni. Tafuta ushauri wa wakili wa uhamiaji kuhusu njia za kisheria zinazofaa hali yako.", "correct_answer_en": "Marrying a Tanzanian citizen does not change citizenship status under GN 487A \\u2014 you will still be treated as a non-citizen. Consult an immigration lawyer about lawful options suitable to your situation.", "answer_type": "procedure", "source_url": "https://www.bowmans.com/jurisdiction/tanzania"}, {"id": "eval_176", "subdomain": "osha_registration", "question_sw": "Kutoka wafanyakazi wangapi OSHA inasema ninahitaji kumwajiri afisa wa usalama mahali pa kazi?", "question_en": "From how many employees does OSHA require me to appoint a safety officer?", "correct_answer_sw": "Mwajiri anahitaji kumwajiri afisa wa usalama akiwa na wafanyakazi zaidi ya 50 kwa mahali pa kazi wa jumla. Kwa tovuti za ujenzi na viwanda, kiwango ni wafanyakazi 20 au zaidi. Thibitisha mahitaji yako maalum na OSHA Tanzania.", "correct_answer_en": "An employer must appoint a safety officer when they have more than 50 employees for general workplaces. For construction and industrial sites the threshold is 20 or more workers. Confirm your specific requirements with OSHA Tanzania.", "answer_type": "number", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_177", "subdomain": "osha_registration", "question_sw": "Ukaguzi wa mahali pa kazi wa OSHA unafanyika mara ngapi kwa mwaka?", "question_en": "How often does OSHA conduct workplace inspections per year?", "correct_answer_sw": "Ukaguzi wa mahali pa kazi unafanyika mara moja kwa mwaka \\u2014 ukaguzi wa kila mwaka ni wa lazima.", "correct_answer_en": "Workplace inspection is conducted once per year \\u2014 the annual inspection is mandatory.", "answer_type": "number", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_178", "subdomain": "osha_registration", "question_sw": "Biashara yangu ina wafanyakazi 2 tu \\u2014 kiwango cha chini cha wafanyakazi kwa usajili wa OSHA ni wangapi?", "question_en": "My business has only 2 employees \\u2014 what is the minimum employee count for OSHA registration?", "correct_answer_sw": "Hakuna kiwango cha chini cha wafanyakazi. OSHA inafunika waajiri wote bila kujali idadi ya wafanyakazi \\u2014 hata wafanyakazi 1 bado unahitaji kusajili.", "correct_answer_en": "There is no minimum employee count. OSHA registration applies to all employers regardless of the number of employees \\u2014 even 1 employee still requires registration.", "answer_type": "number", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_179", "subdomain": "osha_registration", "question_sw": "Duka langu la vifaa lina wafanyakazi 3 tu \\u2014 bado nahitaji kusajili mahali pa kazi na OSHA?", "question_en": "My hardware shop has only 3 employees \\u2014 do I still need to register the workplace with OSHA?", "correct_answer_sw": "Ndiyo. Usajili wa OSHA unahitajika kwa waajiri wote, bila kujali idadi ya wafanyakazi.", "correct_answer_en": "Yes. OSHA registration is required for all employers, regardless of the number of employees.", "answer_type": "yes_no", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_180", "subdomain": "osha_registration", "question_sw": "Nilisajili biashara yangu BRELA tayari \\u2014 hiyo inamaanisha nimesajili OSHA pia?", "question_en": "I already registered my business with BRELA \\u2014 does that mean I am also registered with OSHA?", "correct_answer_sw": "Hapana. Usajili wa BRELA na OSHA ni tofauti kabisa. BRELA inasajili biashara, OSHA inasajili mahali pa kazi. Lazima ufanye usajili wote wawili.", "correct_answer_en": "No. BRELA and OSHA registrations are completely separate. BRELA registers the business, OSHA registers the workplace. You must complete both registrations.", "answer_type": "yes_no", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_181", "subdomain": "osha_registration", "question_sw": "Mahali pa kazi lazima kuwe na cheti halali cha usajili wa OSHA?", "question_en": "Must the workplace have a valid OSHA registration certificate?", "correct_answer_sw": "Ndiyo. Mahali pa kazi lazima kuwe na cheti halali cha usajili wa OSHA.", "correct_answer_en": "Yes. The workplace must hold a valid OSHA registration certificate.", "answer_type": "yes_no", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_182", "subdomain": "osha_registration", "question_sw": "Ukaguzi wa OSHA wa kila mwaka ni wa lazima au wa hiari?", "question_en": "Is the annual OSHA workplace inspection mandatory or optional?", "correct_answer_sw": "Ni wa lazima. Ukaguzi wa kila mwaka wa mahali pa kazi ni sharti la OSHA.", "correct_answer_en": "It is mandatory. The annual workplace inspection is an OSHA requirement.", "answer_type": "yes_no", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_183", "subdomain": "osha_registration", "question_sw": "Mwajiri asiyesajili OSHA anaweza kufungiwa biashara yake?", "question_en": "Can an employer who fails to register with OSHA face closure of their premises?", "correct_answer_sw": "Ndiyo. Adhabu kwa kushindwa kusajili OSHA inaweza kujumuisha faini na kufungwa kwa mahali pa kazi.", "correct_answer_en": "Yes. The penalty for failing to register with OSHA can include a fine and possible closure of the premises.", "answer_type": "yes_no", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_184", "subdomain": "osha_registration", "question_sw": "Nianzishe wapi kusajili mahali pa kazi na OSHA \\u2014 hatua za msingi ni zipi?", "question_en": "Where do I start to register my workplace with OSHA \\u2014 what are the basic steps?", "correct_answer_sw": "Mwajiri lazima asajili mahali pake pa kazi na OSHA, apate cheti cha usajili, na ahakikishe ukaguzi wa kila mwaka unafanyika. Wasiliana na OSHA moja kwa moja kwa hatua kamili za usajili.", "correct_answer_en": "The employer must register the workplace with OSHA, obtain a registration certificate, and ensure the annual inspection takes place. Contact OSHA directly for the complete registration steps.", "answer_type": "procedure", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_185", "subdomain": "osha_registration", "question_sw": "Tofauti kati ya usajili wa BRELA na usajili wa OSHA ni nini \\u2014 nianze na ipi kwanza?", "question_en": "What is the difference between BRELA and OSHA registration \\u2014 which should I do first?", "correct_answer_sw": "BRELA inasajili biashara yenyewe (uhalali wa kibiashara). OSHA inasajili mahali pa kazi (usalama wa wafanyakazi). Hizi ni usajili tofauti na zote mbili zinahitajika. Kawaida BRELA inafanywa kwanza kisha OSHA.", "correct_answer_en": "BRELA registers the business itself (legal entity). OSHA registers the workplace (worker safety). These are separate registrations and both are required. BRELA is typically done first, then OSHA.", "answer_type": "procedure", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_186", "subdomain": "osha_registration", "question_sw": "Nina wafanyakazi 60 na nimesajili OSHA \\u2014 je, pia ninahitaji kumwajiri afisa wa usalama?", "question_en": "I have 60 employees and have registered with OSHA \\u2014 do I also need to appoint a safety officer?", "correct_answer_sw": "Ndiyo. Kwa kuwa una wafanyakazi zaidi ya 50, sheria inakuhitaji kumwajiri afisa wa usalama, mbali na usajili wa kawaida wa OSHA.", "correct_answer_en": "Yes. Since you have more than 50 employees, the law requires you to appoint a safety officer, in addition to the standard OSHA registration.", "answer_type": "procedure", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_187", "subdomain": "osha_registration", "question_sw": "Nimekuwa na biashara kwa miaka 3 bila kusajili OSHA \\u2014 nifanye nini sasa?", "question_en": "I have been operating for 3 years without OSHA registration \\u2014 what should I do now?", "correct_answer_sw": "Unapaswa kusajili mahali pako pa kazi na OSHA haraka iwezekanavyo. Kushindwa kusajili kunaweza kusababisha faini na kufungwa kwa biashara yako.", "correct_answer_en": "You should register your workplace with OSHA as soon as possible. Failure to register can result in a fine and possible closure of the business.", "answer_type": "procedure", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_188", "subdomain": "osha_registration", "question_sw": "Adhabu za kushindwa kusajili mahali pa kazi na OSHA ni zipi?", "question_en": "What are the penalties for failing to register a workplace with OSHA?", "correct_answer_sw": "Adhabu za kushindwa kusajili OSHA zinaweza kujumuisha faini na kufungwa kwa mahali pa kazi.", "correct_answer_en": "Penalties for failing to register with OSHA can include a fine and possible closure of the premises.", "answer_type": "penalty", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_189", "subdomain": "osha_registration", "question_sw": "OSHA ikigundua biashara yangu haijasajiliwa \\u2014 wanaweza kufunga duka langu?", "question_en": "If OSHA discovers my business is not registered \\u2014 can they close my shop?", "correct_answer_sw": "Kufungwa kwa mahali pa kazi ni miongoni mwa hatua za adhabu zinazowezekana. OSHA inaweza kuchukua hatua za utekelezaji, ikiwemo kufunga mahali pa kazi.", "correct_answer_en": "Closure of the premises is among the possible enforcement actions. OSHA can take enforcement steps, including closing the workplace.", "answer_type": "penalty", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_190", "subdomain": "osha_registration", "question_sw": "Kampuni yangu ina wafanyakazi 70 na haina afisa wa usalama \\u2014 hii ni ukiukwaji wa OSHA?", "question_en": "My company has 70 employees and no safety officer \\u2014 is this an OSHA violation?", "correct_answer_sw": "Ndiyo. Kampuni yenye wafanyakazi 70 inahitajika na sheria kumwajiri afisa wa usalama \\u2014 kiwango cha wafanyakazi 50 kinatumika kwa mahali pa kazi wa jumla. Kutomwajiri ni ukiukwaji wa OSHA na unaweza kusababisha adhabu ikiwemo faini na kufungwa kwa biashara. Thibitisha na OSHA Tanzania.", "correct_answer_en": "Yes. An employer with 70 employees is required by law to appoint a safety officer \\u2014 the 50-employee threshold applies to general workplaces. Failure to appoint is an OSHA violation and can result in penalties including fines and business closure. Confirm with OSHA Tanzania.", "answer_type": "penalty", "source_url": "https://auditaxinternational.co.tz/key-tax-implications-of-the-tanzania-finance-act/"}, {"id": "eval_191", "subdomain": "out_of_corpus", "question_sw": "PAYE ya mfanyakazi wangu mwenye mshahara wa TZS 800,000 kwa mwezi ni kiasi gani?", "question_en": "How much PAYE does my employee earning TZS 800,000 per month owe?", "correct_answer_sw": "Swali hili liko nje ya maarifa yangu ya sasa. Tafadhali wasiliana na mshauri wa kodi aliyesajiliwa.", "correct_answer_en": "This question is outside my current knowledge. Please consult a registered tax adviser.", "answer_type": "out_of_corpus_refusal", "source_url": "https://taxsummaries.pwc.com/tanzania/individual/other-taxes"}, {"id": "eval_192", "subdomain": "out_of_corpus", "question_sw": "Kodi ya uagizaji wa simu za mkononi kutoka China ni asilimia ngapi?", "question_en": "What is the import duty rate on mobile phones imported from China?", "correct_answer_sw": "Swali hili liko nje ya maarifa yangu ya sasa. Tafadhali wasiliana na mshauri wa kodi aliyesajiliwa.", "correct_answer_en": "This question is outside my current knowledge. Please consult a registered tax adviser.", "answer_type": "out_of_corpus_refusal", "source_url": "https://taxsummaries.pwc.com/tanzania/individual/other-taxes"}, {"id": "eval_193", "subdomain": "out_of_corpus", "question_sw": "Niliuza ardhi yangu kwa faida \\u2014 kodi ya mapato ya mtaji (capital gains tax) ni kiasi gani nalazima nilipe?", "question_en": "I sold my land at a profit \\u2014 how much capital gains tax do I owe?", "correct_answer_sw": "Swali hili liko nje ya maarifa yangu ya sasa. Tafadhali wasiliana na mshauri wa kodi aliyesajiliwa.", "correct_answer_en": "This question is outside my current knowledge. Please consult a registered tax adviser.", "answer_type": "out_of_corpus_refusal", "source_url": "https://taxsummaries.pwc.com/tanzania/individual/other-taxes"}, {"id": "eval_194", "subdomain": "out_of_corpus", "question_sw": "Kampuni yangu mama iko India \\u2014 mahitaji ya bei ya uhamisho (transfer pricing) Tanzania ni yapi?", "question_en": "My parent company is in India \\u2014 what are Tanzania\'s transfer pricing requirements?", "correct_answer_sw": "Swali hili liko nje ya maarifa yangu ya sasa. Tafadhali wasiliana na mshauri wa kodi aliyesajiliwa.", "correct_answer_en": "This question is outside my current knowledge. Please consult a registered tax adviser.", "answer_type": "out_of_corpus_refusal", "source_url": "https://taxsummaries.pwc.com/tanzania/individual/other-taxes"}, {"id": "eval_195", "subdomain": "out_of_corpus", "question_sw": "Biashara yangu iko Zanzibar \\u2014 sheria za kodi za Zanzibar ni tofauti na Tanzania bara?", "question_en": "My business is in Zanzibar \\u2014 are Zanzibar tax laws different from mainland Tanzania?", "correct_answer_sw": "Swali hili liko nje ya maarifa yangu ya sasa. Tafadhali wasiliana na mshauri wa kodi aliyesajiliwa.", "correct_answer_en": "This question is outside my current knowledge. Please consult a registered tax adviser.", "answer_type": "out_of_corpus_refusal", "source_url": "https://taxsummaries.pwc.com/tanzania/individual/other-taxes"}, {"id": "eval_196", "subdomain": "out_of_corpus", "question_sw": "Nilifanya faida kwa kununua na kuuza Bitcoin Tanzania \\u2014 je, lazima nilipe kodi?", "question_en": "I made a profit buying and selling Bitcoin in Tanzania \\u2014 do I need to pay tax?", "correct_answer_sw": "Swali hili liko nje ya maarifa yangu ya sasa. Tafadhali wasiliana na mshauri wa kodi aliyesajiliwa.", "correct_answer_en": "This question is outside my current knowledge. Please consult a registered tax adviser.", "answer_type": "out_of_corpus_refusal", "source_url": "https://taxsummaries.pwc.com/tanzania/individual/other-taxes"}, {"id": "eval_197", "subdomain": "out_of_corpus", "question_sw": "Thamani ya ardhi yangu inakadiriwa vipi kwa ajili ya stamp duty Tanzania?", "question_en": "How is land valued for stamp duty purposes in Tanzania?", "correct_answer_sw": "Swali hili liko nje ya maarifa yangu ya sasa. Tafadhali wasiliana na mshauri wa kodi aliyesajiliwa.", "correct_answer_en": "This question is outside my current knowledge. Please consult a registered tax adviser.", "answer_type": "out_of_corpus_refusal", "source_url": "https://taxsummaries.pwc.com/tanzania/individual/other-taxes"}, {"id": "eval_198", "subdomain": "out_of_corpus", "question_sw": "Kampuni yangu ya uchimbaji madini \\u2014 asilimia ngapi ya royalty inayohusu dhahabu Tanzania?", "question_en": "My mining company \\u2014 what is the royalty rate for gold mining in Tanzania?", "correct_answer_sw": "Swali hili liko nje ya maarifa yangu ya sasa. Tafadhali wasiliana na mshauri wa kodi aliyesajiliwa.", "correct_answer_en": "This question is outside my current knowledge. Please consult a registered tax adviser.", "answer_type": "out_of_corpus_refusal", "source_url": "https://taxsummaries.pwc.com/tanzania/individual/other-taxes"}, {"id": "eval_199", "subdomain": "out_of_corpus", "question_sw": "Kampuni yangu ya bima \\u2014 kiwango cha ushuru wa malipo ya bima (insurance premium levy) Tanzania ni kiasi gani?", "question_en": "My insurance company \\u2014 what is the insurance premium levy rate in Tanzania?", "correct_answer_sw": "Swali hili liko nje ya maarifa yangu ya sasa. Tafadhali wasiliana na mshauri wa kodi aliyesajiliwa.", "correct_answer_en": "This question is outside my current knowledge. Please consult a registered tax adviser.", "answer_type": "out_of_corpus_refusal", "source_url": "https://taxsummaries.pwc.com/tanzania/individual/other-taxes"}, {"id": "eval_200", "subdomain": "out_of_corpus", "question_sw": "Kampuni yangu iko katika eneo huru la usindikaji wa bidhaa (EPZ) \\u2014 masharti ya kodi ni yapi?", "question_en": "My company is in an export processing zone (EPZ) \\u2014 what are the tax conditions?", "correct_answer_sw": "Swali hili liko nje ya maarifa yangu ya sasa. Tafadhali wasiliana na mshauri wa kodi aliyesajiliwa.", "correct_answer_en": "This question is outside my current knowledge. Please consult a registered tax adviser.", "answer_type": "out_of_corpus_refusal", "source_url": "https://taxsummaries.pwc.com/tanzania/individual/other-taxes"}]')
print(f"[eval] Loaded {len(eval_questions)} eval questions")


In [ ]:
# SYSTEM_PROMPT loaded from chike_config.json in Cell 1

REFUSAL_PHRASES = [
    "nje ya maarifa yangu",
    "n je ya maarifa yangu",       # tokenization variant
    "swali hili liko nje",
    "swali hili liko n je",        # tokenization variant
    "liko njaani za mipaka",
    "liko n jaani za mipaka",      # tokenization variant
    "nje ya mada yangu",
    "haiko katika mada",
    "sina uhakika",
    "sijui",
    "siwezi kusaidia na hili",
    "ninasaidia tu na",
    "tanzania bara tu",
    "mshauri wa kodi",
    "outside my current knowledge",
    "i don't know",
    "i am not sure",
    "consult a registered",
    "beyond my knowledge",
]

def check_refusal(text):
    """Collapse multiple spaces before matching to catch tokenization artifacts like 'n je'."""
    normalized = " ".join(text.lower().split())
    return any(phrase in normalized for phrase in REFUSAL_PHRASES)

def generate_answer(question_sw):
    prompt = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>

"
        f"{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>

"
        f"{question_sw}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>

"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, temperature=1.0, repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def extract_numbers(text):
    text = text.lower()
    nums = set()
    for m in re.findall(r'asilimia\s*(\d+(?:\.\d+)?)', text):
        nums.add(m)
    for m in re.findall(r'(\d+(?:\.\d+)?)\s*%', text):
        nums.add(m)
    for m in re.findall(r'tzs\s*([\d,]+)', text):
        nums.add(m.replace(",", ""))
    for m in re.findall(r'(\d+(?:,\d+)*(?:\.\d+)?)\s*(?:million|m)', text):
        nums.add(m.replace(",", ""))
    for m in re.findall(r'(\d{3,}(?:,\d+)*)', text):
        nums.add(m.replace(",", ""))
    return nums

def score_question(q, generated):
    gen_lower = generated.lower()
    atype     = q.get("answer_type", "")
    correct_sw = q.get("correct_answer_sw", "").lower()
    correct_en = q.get("correct_answer_en", "").lower()

    if atype == "out_of_corpus_refusal":
        return check_refusal(generated)

    if atype in ("number", "penalty"):
        correct_nums = extract_numbers(correct_sw) | extract_numbers(correct_en)
        if not correct_nums:
            return len(gen_lower) > 10
        gen_nums = extract_numbers(generated)
        return len(correct_nums & gen_nums) >= 1

    if atype == "yes_no":
        yes_sw = any(w in correct_sw for w in ["ndiyo","yes","ndio","inaweza","lazima"])
        no_sw  = any(w in correct_sw for w in ["hapana","no","haitakiwi","haihitajiki","haiwezi"])
        gen_yes = any(w in gen_lower for w in ["ndiyo","yes","ndio","inaweza","lazima"])
        gen_no  = any(w in gen_lower for w in ["hapana","no","haitakiwi","haihitajiki","haiwezi"])
        if yes_sw:
            return gen_yes
        if no_sw:
            return gen_no
        return len(gen_lower) > 10

    if atype in ("definition", "procedure"):
        words_sw = set(w for w in correct_sw.split() if len(w) > 4)
        words_en = set(w for w in correct_en.split() if len(w) > 4)
        all_words = words_sw | words_en
        if not all_words:
            return len(gen_lower) > 20
        gen_words = set(gen_lower.split())
        return len(all_words & gen_words) >= 3

    return len(gen_lower) > 20

print("[eval] Scoring functions ready")


In [ ]:
results = []
print(f"[eval] Starting inference on {len(eval_questions)} questions ...")

for i, q in enumerate(eval_questions):
    try:
        generated = generate_answer(q["question_sw"])
        passed = score_question(q, generated)
    except Exception as e:
        generated = f"ERROR: {e}"
        passed = False

    results.append({
        "id":              q["id"],
        "subdomain":       q["subdomain"],
        "answer_type":     q.get("answer_type", ""),
        "question_sw":     q["question_sw"],
        "correct_answer_sw": q["correct_answer_sw"],
        "generated":       generated,
        "pass":            passed,
    })

    if (i + 1) % 20 == 0 or i == 0:
        rp = sum(r["pass"] for r in results)
        print(f"  [{i+1}/{len(eval_questions)}] running accuracy: {rp}/{i+1} = {rp/(i+1):.1%}")

print("[eval] Inference complete")


In [ ]:
from collections import defaultdict

subdomain_scores = defaultdict(lambda: {"pass": 0, "total": 0})
for r in results:
    subdomain_scores[r["subdomain"]]["total"] += 1
    if r["pass"]:
        subdomain_scores[r["subdomain"]]["pass"] += 1

in_corpus = [r for r in results if r["subdomain"] != "out_of_corpus"]
refusal   = [r for r in results if r["subdomain"] == "out_of_corpus"]

in_corpus_pass  = sum(r["pass"] for r in in_corpus)
in_corpus_total = len(in_corpus)
refusal_pass    = sum(r["pass"] for r in refusal)
refusal_total   = len(refusal)

in_corpus_acc = in_corpus_pass / in_corpus_total if in_corpus_total > 0 else 0
refusal_acc   = refusal_pass   / refusal_total   if refusal_total   > 0 else 0
acc_gate_pass = in_corpus_acc > 0.85
ref_gate_pass = refusal_acc   > 0.70
gate_passed   = acc_gate_pass and ref_gate_pass

print("\n========================================")
print("AFRICA-GIANTS ACCURACY GATE RESULTS")
print("========================================")
print("\nBy subdomain:")
for sd in sorted(subdomain_scores.keys()):
    s = subdomain_scores[sd]
    pct = s["pass"] / s["total"] if s["total"] > 0 else 0
    bar = "*" * int(pct * 20)
    print(f"  {sd:<30} {s['pass']:>3}/{s['total']:<3} = {pct:>5.1%}  {bar}")

print()
print(f"In-corpus accuracy:    {in_corpus_pass}/{in_corpus_total} = {in_corpus_acc:.1%}   >85% {'PASS' if acc_gate_pass else 'FAIL'}")
print(f"Out-of-corpus refusal: {refusal_pass}/{refusal_total} = {refusal_acc:.1%}   >70% {'PASS' if ref_gate_pass else 'FAIL'}")
print()
if gate_passed:
    print("*** GATE PASSED ***")
else:
    print("GATE FAILED — both >85% in-corpus AND >70% refusal required")
print("========================================")


In [ ]:
gate_result = {
    "timestamp":           datetime.now(timezone.utc).isoformat(),
    "model":               ADAPTER_REPO,
    "base_model":          BASE_MODEL,
    "in_corpus_correct":   in_corpus_pass,
    "in_corpus_total":     in_corpus_total,
    "in_corpus_accuracy":  round(in_corpus_acc, 4),
    "in_corpus_pass":      acc_gate_pass,
    "refusal_correct":     refusal_pass,
    "refusal_total":       refusal_total,
    "refusal_accuracy":    round(refusal_acc, 4),
    "refusal_pass":        ref_gate_pass,
    "gate_passed":         gate_passed,
    "by_subdomain": {
        sd: {
            "pass":     v["pass"],
            "total":    v["total"],
            "accuracy": round(v["pass"] / v["total"], 4) if v["total"] > 0 else 0,
        }
        for sd, v in sorted(subdomain_scores.items())
    },
    "per_question": results,
}

with open("/kaggle/working/gate_001_results.json", "w", encoding="utf-8") as f:
    json.dump(gate_result, f, indent=2, ensure_ascii=False)
print("[results] Saved /kaggle/working/gate_001_results.json")


In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=hf_token)
api.upload_file(
    path_or_fileobj="/kaggle/working/gate_001_results.json",
    path_in_repo="gate_001_results.json",
    repo_id=ADAPTER_REPO,
    repo_type="model",
    token=hf_token,
)
print(f"[results] Uploaded gate_001_results.json to {ADAPTER_REPO}")
print("Done.")
